In [ ]:
from capture_utils_v2 import CaptureSystem

# Initialize the capture system
system = CaptureSystem()


In [ ]:
system.display_drawer()


In [ ]:
system.run_aruco_detector()
# system.run_aruco_detector_manual()  # Instead of run_aruco_detector()

In [ ]:
ret = system.photometric_calibration(return_captured=True)
# system.photometric_calibration_adaptive()

In [ ]:
# save system orig_proj_corners and corners_img_proj to a file
import pickle
with open("capture_system_state.pkl", "wb") as f:
    pickle.dump({'orig_proj_corners': system.orig_proj_corners,
                  'corners_img_proj': system.corners_img_proj, 
                  'orig_img': system.orig_img, 
                  'img_non_zero_section': system.img_non_zero_section, 
                  'H': system.H
                  }, f)

## Infer

In [ ]:
# for capaa
# import sys
# sys.path.append(r'C:\git\CAPAA\src\python')
# from utils import init_prj_window

In [ ]:
from capture_utils_v2 import CaptureSystem
import numpy as np
import cv2
import pickle
from aruco_pose import get_camera_angles_from_frame
# Initialize the capture system


In [ ]:
im_path = r"C:\git\PhysicalAdverserialProj\results\ensamble_classifier_2025-12-24_07_11_top_patches\2_0.54.png"
# im_path = r"C:\git\PhysicalAdverserialProj\results\best_patch_ensamble_classifier_16x16_6_2025-12-27_10_24.png"
# im_path = './results/best_patch_ensamble_classifier_16x16_16_2025-12-24_10_08.png'
# im_path = r'C:\git\PhysicalAdverserialProj\results\rejuv_MobileNetV3_16x16_2026-01-18_14_56_top_patches\5_1.0.png'
# im_path = r"C:\git\PhysicalAdverserialProj\results\MobileNetV3_16x16_2026-01-20_10_58_top_patches\33_0.86.png"
# im_path = r"C:\git\PhysicalAdverserialProj\results\MobileNetV3_16x16_2026-01-20_15_43_top_patches\1_1.0.png"
# im_path = r"C:\git\PhysicalAdverserialProj\results\MobileNetV3_16x16_2026-01-20_16_26_top_patches\2_1.0.png"
image_to_project =  cv2.imread(im_path)
image_to_project = cv2.cvtColor(image_to_project, cv2.COLOR_BGR2RGB)

In [ ]:
# import pickle

# system = CaptureSystem()

# # with open("capture_system_state_ablation_base_backup.pkl", "rb") as f:
# with open("capture_system_state.pkl", "rb") as f:
#     system_dict = pickle.load(f)


# system.orig_proj_corners, system.corners_img_proj, system.orig_img, system.img_non_zero_section \
#     = system_dict['orig_proj_corners'], system_dict['corners_img_proj'],system_dict['orig_img'], system_dict['img_non_zero_section']

In [ ]:
system.plot_on_screen(image_to_project)

In [ ]:
#capaa
# from omegaconf import DictConfig, OmegaConf
# import matplotlib as plt
# plt.use('Qt5Agg')
# p = r"C:\git\CAPAA\src\python\data\setups\jeep8\prj\adv\CAPAA_PCNet_l1+ssim_500_24_2000\camdE_caml2\5\classifier_all\img_0001.png"
# img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
# setup_info = DictConfig(dict(
#     prj_screen_sz      = (1920 , 1080),   # projector screen resolution (i.e., set in the OS)
#     prj_im_sz          = (256 , 256),   # projector input image resolution, the image will be scaled to match the prj_screen_sz by plt
#     prj_offset         = (2560,   0),   # an offset to move the projector plt figure to the correct screen (check your OS display setting)
#     prj_brightness     = 0.5,           # brightness (0~1 float) of the projector gray image for scene illumination.

#     # adjust the two params below according to your ProCams latency, until the projected and captured numbers images are correctly matched
#     delay_frames = 5,  # how many frames to drop before we capture the correct one, increase it when ProCams are not in sync
#     delay_time = 0.1,  # a delay time (s) between the project and capture operations for software sync, increase it when ProCams are not in sync
#     )
# )
# prj = init_prj_window(*setup_info['prj_screen_sz'], 0.5, setup_info['prj_offset'])
# prj.set_data(img)

In [ ]:
import torch
# from classfier import predict_raw, weights
# from classfier_ensemble import predict_raw, weights
# from classfier_ensemble_v2 import predict_raw, weights
# from classfier_dino import predict_raw, weights
# from classfier_mobilenet import predict_raw, weights
# from classfier_dino import predict_raw, weights
# from classfier_ensemble import predict_raw as predict_raw_ens
# from classfier_mobilenet import weights

# weights_dict = {
#     'inception': 1.0,
#     'resnet': 0.0,
#     'vgg': 0.0,
#     'vit': 0.0,
#     'dino': 0.0
# }

# predict_raw = lambda x: predict_raw_ens(x, weights_dict)

In [ ]:
from classfier_mobilenet import predict_raw, weights

In [ ]:
weights.meta['categories'].index('water bottle')

In [ ]:
forbiden_classes = [898]

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def add_text_with_background(img, text, position, font_size=24, text_color=(255, 255, 255), bg_color=(0, 0, 0)):
    """Add text with background using PIL (supports Arial font)"""
    # Convert BGR to RGB for PIL
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    
    # Try to load Arial font, fall back to default if not available
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        try:
            font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", font_size)
        except:
            font = ImageFont.load_default()
    
    # Get text bounding box
    bbox = draw.textbbox(position, text, font=font)
    padding = 5
    
    # Draw black background rectangle
    draw.rectangle([bbox[0] - padding, bbox[1] - padding, 
                    bbox[2] + padding, bbox[3] + padding], fill=bg_color)
    
    # Draw white text
    draw.text(position, text, font=font, fill=text_color)
    
    # Convert back to BGR for OpenCV
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()
cap = system.cap

caps = []
caps_with_text = []
results = []
raw_predictions = []
for i in range(2400):
    r = cap.read()[1]
    tr = tt(r)
    with torch.no_grad():
        p = predict_raw(tr.unsqueeze(0).cuda())
        res = weights.meta["categories"][p[0].argmax(0).item()]
        prob = p[0].max(0).values.item() * 100
        raw_predictions.append(p)
    
    # add text
    r_orig = r.copy()
    pose_result = get_camera_angles_from_frame(r)
    
    # White text on black background with Arial font
    r = add_text_with_background(r, f'Pred: {res}: {prob:.2f}%', (10, 10), font_size=28)
    
    # Display camera angles if ArUco marker is detected
    # if pose_result['found']:
    # angle = pose_result['angle']
    # dist = pose_result['distance_m']
        # cv2.putText(r, f'H: {angle:.1f}deg  Dist: {dist:.1f}m', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    # else:
    #     r = add_text_with_background(r, 'ArUco not detected', (10, 50), font_size=22, text_color=(255, 100, 100))
    
    cv2.imshow('frame', r)
    results.append(res)

    caps_with_text.append(r)
    caps.append(r_orig)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        raise

In [ ]:
# save caps to disk

In [ ]:
import os
import datetime
cur_time = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
infer_caps_dir = r"C:\git\PhysicalAdverserialProj\infer_caps_dir"
exp_name = rf"{os.path.basename(im_path).split('.')[0]}"
# exp_name = "capaa_jeep8_5_1"
os.makedirs(f"{infer_caps_dir}/{exp_name}/{cur_time}/")
with open(f"{infer_caps_dir}/{exp_name}/{cur_time}/caps.pkl", "wb") as f:
    pickle.dump(caps, f)
with open(f"{infer_caps_dir}/{exp_name}/{cur_time}/caps_with_text.pkl", "wb") as f:
    pickle.dump(caps_with_text, f)
with open(f"{infer_caps_dir}/{exp_name}/{cur_time}/results.pkl", "wb") as f:
    pickle.dump(results, f)
with open(f"{infer_caps_dir}/{exp_name}/{cur_time}/raw_predictions.pkl", "wb") as f:
    pickle.dump(raw_predictions, f)

In [ ]:
cv2.destroyAllWindows()

### With tracking

### Tracking with Rotation Compensation
Uses SIFT tracking to adjust the existing projection (from `system.plot_on_screen`) to compensate for object movement and rotation.

In [ ]:
# Optional: Load ProCam stereo calibration for future 3D-aware projection
# For now, we use the existing system calibration (system.orig_proj_corners, etc.)

import pickle
import os
import numpy as np

CALIB_DIR = r"C:\git\PhysicalAdverserialProj\procam_calibration_data"
calib_file = os.path.join(CALIB_DIR, 'procam_calibration_aruco.pkl')

if os.path.exists(calib_file):
    with open(calib_file, 'rb') as f:
        procam_calib = pickle.load(f)
    print(f"✓ ProCam calibration loaded from {calib_file}")
    print(f"  Stereo RMS error: {procam_calib.get('stereo_rms_error', 'N/A'):.4f} px")
    print(f"  Baseline: {np.linalg.norm(procam_calib['T']):.1f} mm")
else:
    print(f"⚠ ProCam calibration not found at {calib_file}")
    print("  Using system's photometric calibration only")
    procam_calib = None

In [ ]:
# Load all calibration from procam_calibration_aruco.pkl
camera_matrix = procam_calib['camera_matrix'].copy()
camera_dist = procam_calib['camera_dist']
projector_matrix = procam_calib['projector_matrix']
projector_dist = procam_calib.get('projector_dist', np.zeros(5))
R_stereo = procam_calib['R']  # Rotation from camera to projector
T_stereo = procam_calib['T']  # Translation from camera to projector

PROJECTOR_X_OFFSET = 1920
PRJ_W, PRJ_H = system.screen_res


In [ ]:
# from classfier import *
# from tracking_utils import TrackerSystem
# image_to_project =  patch = cv2.imread(r'C:\git\PhysicalAdverserialProj\results\best_patch_16x16_1_2025-12-13_16_22.png')

# tracker = TrackerSystem(system, predict_raw, weights, printed_aruco_id=10)
# caps, results = tracker.track_project_and_classify(image_to_project)

In [ ]:
# ============================================================================
# ARUCO TRACKING + STEREO PROJECTION + CLASSIFICATION
# ============================================================================
# Projects patch that MOVES with the ArUco marker using 3D ProCam stereo calibration.
# Projection follows the marker in real-time, robust to rotations and translations.

import cv2
import cv2.aruco as aruco
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

# Helper functions
tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

def add_text_with_background(img, text, position, font_size=24, text_color=(255,255,255), bg_color=(0,0,0)):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", font_size)
    except:
        font = ImageFont.load_default()
    bbox = draw.textbbox(position, text, font=font)
    padding = 5
    draw.rectangle([bbox[0]-padding, bbox[1]-padding, bbox[2]+padding, bbox[3]+padding], fill=bg_color)
    draw.text(position, text, font=font, fill=text_color)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

# ============================================================================
# CONFIGURATION
# ============================================================================

ARUCO_DICT_TYPE = aruco.DICT_4X4_50
ARUCO_MARKER_SIZE_MM = 61.0  # 6 cm
ARUCO_MARKER_ID = -1  # -1 = any, or specify ID

# IMPORTANT: Reload camera_matrix fresh each time to avoid cumulative scaling
camera_matrix = procam_calib['camera_matrix'].copy()
camera_dist = procam_calib['camera_dist'].copy()

# Check actual frame resolution and scale camera matrix if needed
ret, test_frame = system.cap.read()
if ret:
    actual_h, actual_w = test_frame.shape[:2]
    # Use known calibration resolution (from procam calibration notes)
    # The procam_calib was done at 640x480
    calib_w, calib_h = 640, 480
    
    print(f"Calibration resolution: {calib_w}x{calib_h}")
    print(f"Actual frame resolution: {actual_w}x{actual_h}")
    print(f"Original camera matrix: fx={camera_matrix[0,0]:.1f}, fy={camera_matrix[1,1]:.1f}, cx={camera_matrix[0,2]:.1f}, cy={camera_matrix[1,2]:.1f}")
    
    if actual_w != calib_w or actual_h != calib_h:
        scale_x = actual_w / calib_w
        scale_y = actual_h / calib_h
        print(f"⚠ Resolution mismatch! Scaling camera matrix by ({scale_x:.2f}, {scale_y:.2f})")
        camera_matrix[0, 0] *= scale_x  # fx
        camera_matrix[1, 1] *= scale_y  # fy
        camera_matrix[0, 2] *= scale_x  # cx
        camera_matrix[1, 2] *= scale_y  # cy

print(f"Final camera matrix: fx={camera_matrix[0,0]:.1f}, fy={camera_matrix[1,1]:.1f}, cx={camera_matrix[0,2]:.1f}, cy={camera_matrix[1,2]:.1f}")
print(f"Projector matrix: fx={projector_matrix[0,0]:.1f}, fy={projector_matrix[1,1]:.1f}")


# Smoothing parameters - lower values = more smoothing but more lag
# Higher values = more responsive but more jitter
CORNER_SMOOTH_ALPHA = 0.5   # EMA for detected corners (0.3->0.5 for faster response)
POSE_SMOOTH_ALPHA = 0.4     # EMA for pose (0.2->0.4 for faster response during rotation)

# Patch to project
patch = cv2.imread(im_path)
patch_h, patch_w = patch.shape[:2]
src_corners = np.array([[0,0],[patch_w,0],[patch_w,patch_h],[0,patch_h]], dtype=np.float32)

# ============================================================================
# 3D PATCH CONFIGURATION - Relative to marker center
# ============================================================================
# Similar to debug_aruco_projection.ipynb: define patch position in marker frame.
# Will be computed during initialization by back-projecting system.corners_img_proj.

patch_3d_corners = None  # Set during initialization: 4x3 array in marker coords

def camera_pixel_to_marker_3d(pixel, rvec, tvec):
    """Back-project a camera pixel to 3D on the marker plane (Z=0)."""
    pixel_undist = cv2.undistortPoints(
        np.array([[pixel]], dtype=np.float32),
        camera_matrix, camera_dist, P=camera_matrix
    )[0, 0]
    
    R_cam, _ = cv2.Rodrigues(rvec)
    fx, fy = camera_matrix[0, 0], camera_matrix[1, 1]
    cx, cy = camera_matrix[0, 2], camera_matrix[1, 2]
    
    ray_cam = np.array([
        (pixel_undist[0] - cx) / fx,
        (pixel_undist[1] - cy) / fy,
        1.0
    ])
    
    cam_pos_marker = -R_cam.T @ tvec.flatten()
    ray_marker = R_cam.T @ ray_cam
    
    if abs(ray_marker[2]) < 1e-6:
        return None
    
    lambda_val = -cam_pos_marker[2] / ray_marker[2]
    if lambda_val < 0:
        return None
    
    point_3d = cam_pos_marker + lambda_val * ray_marker
    point_3d[2] = 0
    return point_3d.astype(np.float32)

print("Patch 3D corners will be computed from corners_img_proj during init.")

# ============================================================================
# STEREO PROJECTION FUNCTION
# ============================================================================

def project_to_projector_3d(pts_3d, rvec_cam, tvec_cam):
    """Project 3D object-frame points to projector pixel coordinates via stereo.
    
    Pipeline (matching debug_aruco_projection.ipynb):
      1. Camera extrinsics: R_cam, t_cam from solvePnP (object → camera)
      2. Stereo transform: R_proj = R_st @ R_cam, t_proj = R_st @ t_cam + T_st (object → projector)
      3. projectPoints with projector intrinsics
    """
    R_cam, _ = cv2.Rodrigues(rvec_cam)
    R_proj = R_stereo @ R_cam
    t_proj = (R_stereo @ tvec_cam.flatten() + T_stereo.flatten()).reshape(3, 1)
    rvec_proj, _ = cv2.Rodrigues(R_proj)
    
    pts, _ = cv2.projectPoints(pts_3d, rvec_proj, t_proj, projector_matrix, projector_dist)
    return pts.reshape(-1, 2).astype(np.float32)

# ============================================================================
# ARUCO SETUP
# ============================================================================

aruco_dict = aruco.getPredefinedDictionary(ARUCO_DICT_TYPE)
aruco_params = aruco.DetectorParameters()
aruco_params.cornerRefinementMethod = aruco.CORNER_REFINE_SUBPIX
aruco_params.cornerRefinementWinSize = 5
aruco_params.cornerRefinementMaxIterations = 30
aruco_params.cornerRefinementMinAccuracy = 0.01
aruco_detector = aruco.ArucoDetector(aruco_dict, aruco_params)

# 3D marker corners (OpenCV ArUco convention: Y increases downward)
half_m = ARUCO_MARKER_SIZE_MM / 2.0
marker_obj_points = np.array([
    [-half_m, -half_m, 0],   # top-left
    [ half_m, -half_m, 0],   # top-right
    [ half_m,  half_m, 0],   # bottom-right
    [-half_m,  half_m, 0],   # bottom-left
], dtype=np.float32)

# State for smoothing
smoothed_corners = None
smoothed_rvec = None
smoothed_tvec = None

def detect_aruco_with_pose(frame):
    """Detect ArUco marker and estimate pose. Returns smoothed results."""
    global smoothed_corners, smoothed_rvec, smoothed_tvec
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if len(frame.shape) == 3 else frame
    corners, ids, _ = aruco_detector.detectMarkers(gray)
    
    if ids is None or len(ids) == 0:
        return False, None, None, None, None
    
    idx = 0
    if ARUCO_MARKER_ID >= 0:
        for i, mid in enumerate(ids):
            if mid[0] == ARUCO_MARKER_ID:
                idx = i
                break
        else:
            return False, None, None, None, None
    
    mc = corners[idx].reshape(-1, 2).astype(np.float32)
    mid = int(ids[idx][0])
    
    # Smooth corners first
    if smoothed_corners is None:
        smoothed_corners = mc.copy()
    else:
        smoothed_corners = CORNER_SMOOTH_ALPHA * mc + (1 - CORNER_SMOOTH_ALPHA) * smoothed_corners
    
    # Estimate pose from smoothed corners
    if smoothed_rvec is not None and smoothed_tvec is not None:
        success, rvec, tvec = cv2.solvePnP(
            marker_obj_points, smoothed_corners, camera_matrix, camera_dist,
            rvec=smoothed_rvec.copy(), tvec=smoothed_tvec.copy(),
            useExtrinsicGuess=True, flags=cv2.SOLVEPNP_ITERATIVE
        )
    else:
        # First detection - debug info
        corner_span = np.max(smoothed_corners, axis=0) - np.min(smoothed_corners, axis=0)
        print(f"[DEBUG] Detected corners (2D pixels):\\n{smoothed_corners}")
        print(f"[DEBUG] Corners span: {corner_span[0]:.1f} x {corner_span[1]:.1f} pixels")
        print(f"[DEBUG] marker_obj_points (3D mm):\\n{marker_obj_points}")
        print(f"[DEBUG] Marker size: {ARUCO_MARKER_SIZE_MM} mm")
        print(f"[DEBUG] Camera fx={camera_matrix[0,0]:.1f}, fy={camera_matrix[1,1]:.1f}")
        
        # Expected distance: marker_size * f / pixel_span
        expected_z = ARUCO_MARKER_SIZE_MM * camera_matrix[0,0] / corner_span[0]
        print(f"[DEBUG] Expected Z distance: ~{expected_z:.0f} mm")
        
        # Try ITERATIVE first (more stable than IPPE for moderate angles)
        success, rvec, tvec = cv2.solvePnP(
            marker_obj_points, smoothed_corners, camera_matrix, camera_dist,
            flags=cv2.SOLVEPNP_ITERATIVE
        )
        
        if success:
            print(f"[ITERATIVE] tvec={tvec.flatten()}")
            # Sanity check: Z should be near expected
            z_err = abs(tvec[2, 0] - expected_z) / expected_z
            if z_err > 0.5:  # More than 50% off
                print(f"[WARN] Z={tvec[2,0]:.1f}mm differs from expected {expected_z:.0f}mm by {z_err*100:.0f}%")
                # Try IPPE_SQUARE as fallback
                retval, rvecs, tvecs, _ = cv2.solvePnPGeneric(
                    marker_obj_points, smoothed_corners, camera_matrix, camera_dist,
                    flags=cv2.SOLVEPNP_IPPE_SQUARE
                )
                if retval > 0:
                    # Pick solution closest to expected_z
                    for i, tv in enumerate(tvecs):
                        print(f"[IPPE] Solution {i}: Z={tv[2,0]:.1f}mm")
                        if abs(tv[2,0] - expected_z) < abs(tvec[2,0] - expected_z):
                            tvec = tv
                            rvec = rvecs[i]
                    print(f"[IPPE] Selected Z={tvec[2,0]:.1f}mm")
        
        if not success:
            return False, None, None, None, None
        
        print(f"[Final] rvec={rvec.flatten()}, tvec={tvec.flatten()}")
    
    # Sanity check: Z should be positive (marker in front of camera)
    if tvec[2, 0] < 0:
        # Flip the pose by rotating 180 degrees around X axis
        R, _ = cv2.Rodrigues(rvec)
        R_flip = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]], dtype=np.float64)
        R = R @ R_flip
        rvec, _ = cv2.Rodrigues(R)
        tvec = -tvec
    
    # Check for pose flip (normal pointing wrong direction = Z axis flip)
    R, _ = cv2.Rodrigues(rvec)
    normal = R[:, 2]  # Z-axis of marker in camera frame
    
    if smoothed_rvec is not None:
        R_prev, _ = cv2.Rodrigues(smoothed_rvec)
        normal_prev = R_prev[:, 2]
        # If normal flipped (dot product negative), reject this pose
        if np.dot(normal, normal_prev) < 0:
            # Pose flipped - keep previous pose
            return True, smoothed_rvec, smoothed_tvec, smoothed_corners, mid
    
    # Smooth pose
    if smoothed_rvec is None:
        smoothed_rvec = rvec.copy()
        smoothed_tvec = tvec.copy()
    else:
        smoothed_rvec = POSE_SMOOTH_ALPHA * rvec + (1 - POSE_SMOOTH_ALPHA) * smoothed_rvec
        smoothed_tvec = POSE_SMOOTH_ALPHA * tvec + (1 - POSE_SMOOTH_ALPHA) * smoothed_tvec
    
    return True, smoothed_rvec, smoothed_tvec, smoothed_corners, mid

# ============================================================================
# WINDOW SETUP
# ============================================================================

cv2.namedWindow("Projector", cv2.WND_PROP_FULLSCREEN)
cv2.moveWindow("Projector", PROJECTOR_X_OFFSET, 0)
cv2.setWindowProperty("Projector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.namedWindow("Preview", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Preview", 1280, 960)

# Initial blank projection
proj_img = np.zeros((PRJ_H, PRJ_W, 3), dtype=np.uint8)
cv2.imshow("Projector", proj_img)
cv2.waitKey(100)

# ============================================================================
# TRACKING LOOP (with initialization phase)
# ============================================================================

print("=" * 60)
print("ARUCO TRACKING + STEREO PROJECTION + CLASSIFICATION")
print("=" * 60)
print(f"Stereo baseline: {np.linalg.norm(T_stereo):.1f} mm")
print("=" * 60)

# --- INITIALIZATION PHASE ---
# Project calibrated corners to camera, detect marker, compute 3D patch corners
print("\n--- INITIALIZATION PHASE ---")
print("Projecting calibration pattern and detecting marker...")

# Get calibrated projection corners from photometric calibration
calib_proj_corners = system.orig_proj_corners.reshape(-1, 2).astype(np.float32)
print(f"Calibrated projector corners:\n{calib_proj_corners}")

# DON'T show patch during init - keep blank to avoid jump when tracking starts
# The projection will appear at the correct tracked position after SPACE
proj_img = np.zeros((PRJ_H, PRJ_W, 3), dtype=np.uint8)
cv2.imshow("Projector", proj_img)
cv2.waitKey(100)

# Capture frames until marker is detected stably
init_success = False
init_frames = 0
MAX_INIT_FRAMES = 300

smoothed_corners = None
smoothed_rvec = None
smoothed_tvec = None

while init_frames < MAX_INIT_FRAMES:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    init_frames += 1
    display = frame.copy()
    
    ok, rvec, tvec, mc, mid = detect_aruco_with_pose(frame)
    
    if ok:
        # Draw marker and pose
        cv2.polylines(display, [mc.astype(np.int32)], True, (0, 255, 0), 2)
        
        # DEBUG: Print pose info
        tf = tvec.flatten()
        rf = rvec.flatten()
        R_dbg, _ = cv2.Rodrigues(rvec)
        print(f"[INIT] rvec=({rf[0]:.3f}, {rf[1]:.3f}, {rf[2]:.3f}) tvec=({tf[0]:.1f}, {tf[1]:.1f}, {tf[2]:.1f}) normal=({R_dbg[0,2]:.2f}, {R_dbg[1,2]:.2f}, {R_dbg[2,2]:.2f})")
        
        cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 30)
        
        # Draw where calibrated corners appear in camera
        # Back-project projector corners to camera coordinates
        # We need to find where the projected patch lands in the camera image
        # Use the current corners_img_proj if available, otherwise estimate
        if hasattr(system, 'corners_img_proj') and system.corners_img_proj is not None:
            cam_corners = system.corners_img_proj.reshape(-1, 2).astype(np.int32)
            cv2.polylines(display, [cam_corners], True, (255, 0, 255), 2)
            cv2.putText(display, "Magenta=Calibrated area", (10, 90), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
        
        cv2.putText(display, f"INIT: Marker detected! Frame {init_frames}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display, "Press SPACE to confirm, Q to quit", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    else:
        cv2.putText(display, f"INIT: Looking for marker... Frame {init_frames}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    cv2.imshow("Preview", display)
    key = cv2.waitKey(30) & 0xFF
    
    if key == ord(' ') and ok:
        # User confirmed - compute 3D patch corners in marker frame
        tf = tvec.flatten()
        print(f"\n✓ Marker ID {mid} detected")
        print(f"  tvec = ({tf[0]:.1f}, {tf[1]:.1f}, {tf[2]:.1f}) mm")
        print(f"  Distance from camera: {np.linalg.norm(tf):.1f} mm")
        
        # Sanity check: distance should be reasonable (200-1000mm typical)
        if np.linalg.norm(tf) < 100:
            print(f"⚠ WARNING: Marker distance {np.linalg.norm(tf):.1f}mm seems too close!")
            print(f"  Camera matrix fx,fy = {camera_matrix[0,0]:.1f}, {camera_matrix[1,1]:.1f}")
        
        # Use camera corners from photometric calibration
        if hasattr(system, 'corners_img_proj') and system.corners_img_proj is not None:
            cam_corners = system.corners_img_proj.reshape(-1, 2)
        else:
            print("⚠ corners_img_proj not available, using estimated positions")
            # Fallback: project projector corners through stereo calibration (inverse)
            cam_corners = calib_proj_corners  # This won't be accurate but prevents crash
        
        # Back-project each camera corner to 3D on marker plane
        patch_3d_corners_marker = []
        for i, cam_pt in enumerate(cam_corners):
            pt_3d = camera_pixel_to_marker_3d(cam_pt, rvec, tvec)
            if pt_3d is not None:
                patch_3d_corners_marker.append(pt_3d)
                print(f"  Corner {i}: camera ({cam_pt[0]:.0f}, {cam_pt[1]:.0f}) -> 3D ({pt_3d[0]:.1f}, {pt_3d[1]:.1f}, {pt_3d[2]:.1f}) mm")
            else:
                print(f"  Corner {i}: FAILED to back-project!")
        
        if len(patch_3d_corners_marker) == 4:
            patch_3d_corners_marker = np.array(patch_3d_corners_marker, dtype=np.float32)
            # Compute patch dimensions
            width = np.linalg.norm(patch_3d_corners_marker[1] - patch_3d_corners_marker[0])
            height = np.linalg.norm(patch_3d_corners_marker[3] - patch_3d_corners_marker[0])
            center = patch_3d_corners_marker.mean(axis=0)
            print(f"\n✓ Patch 3D corners computed:")
            print(f"  Size: {width:.1f} x {height:.1f} mm")
            print(f"  Center: ({center[0]:.1f}, {center[1]:.1f}) mm from marker")
            print(f"  3D corners (marker frame):\n{patch_3d_corners_marker}")
            
            # Test projection: project these corners back to projector
            test_proj = project_to_projector_3d(patch_3d_corners_marker, rvec, tvec)
            print(f"  Test projection to projector: {test_proj.tolist()}")
            print(f"  Projector size: {PRJ_W}x{PRJ_H}")
            print(f"  Original calib_proj_corners: {calib_proj_corners.tolist()}")
            
            # Also test: project the MARKER corners (should work like debug notebook)
            marker_proj = project_to_projector_3d(marker_obj_points, rvec, tvec)
            print(f"  Marker corners projection: {marker_proj.tolist()}")
            
            init_success = True
            break
        else:
            print("⚠ Could not compute all 4 corners. Try again.")
    
    elif key == ord('q'):
        break

if not init_success:
    print("\n✗ Initialization failed!")
    cv2.destroyAllWindows()
    raise RuntimeError("Could not initialize - marker not detected or corners not computed")

# --- TRACKING PHASE ---
print("\n--- TRACKING PHASE ---")
print("  q=quit | d=toggle debug")
print("=" * 60)

caps = []
caps_with_text = []
results = []
raw_predictions = []

frame_idx = 1
fps_start = cv2.getTickCount()
fps = 0
show_debug = True
lost_count = 0
last_proj_corners = None

try:
    while True:
        ret, frame = system.cap.read()
        if not ret:
            continue
        
        display = frame.copy()
        ok, rvec, tvec, mc, mid = detect_aruco_with_pose(frame)
        
        if ok:
            lost_count = 0
            
            # Project patch 3D corners to projector coordinates
            proj_corners = project_to_projector_3d(patch_3d_corners_marker, rvec, tvec)
            last_proj_corners = proj_corners
            
            # Debug: print projected corners every 60 frames
            if frame_idx % 60 == 1:
                print(f"Frame {frame_idx}: proj_corners = {proj_corners.tolist()}")
                print(f"  Bounds check: x=[{proj_corners[:,0].min():.0f}, {proj_corners[:,0].max():.0f}], y=[{proj_corners[:,1].min():.0f}, {proj_corners[:,1].max():.0f}]")
                print(f"  Projector size: {PRJ_W}x{PRJ_H}")
            
            # Warp patch to projector
            M = cv2.getPerspectiveTransform(src_corners, proj_corners)
            proj_img = cv2.warpPerspective(patch, M, (PRJ_W, PRJ_H))
            
            if show_debug and mc is not None:
                # Draw detected ArUco corners (green)
                cv2.polylines(display, [mc.astype(np.int32)], True, (0, 255, 0), 2)
                for i, c in enumerate(mc):
                    cv2.circle(display, tuple(c.astype(int)), 4, (0, 255, 0), -1)
                
                # Draw pose axes with debug
                tf_dbg = tvec.flatten()
                rf_dbg = rvec.flatten()
                R_dbg, _ = cv2.Rodrigues(rvec)
                if frame_idx % 30 == 1:
                    print(f"[TRACK] rvec=({rf_dbg[0]:.3f}, {rf_dbg[1]:.3f}, {rf_dbg[2]:.3f}) tvec=({tf_dbg[0]:.1f}, {tf_dbg[1]:.1f}, {tf_dbg[2]:.1f}) norm=({R_dbg[0,2]:.2f}, {R_dbg[1,2]:.2f}, {R_dbg[2,2]:.2f})")
                cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 20)
                
                # Draw expected patch location in camera frame (cyan)
                cam_patch_pts, _ = cv2.projectPoints(patch_3d_corners_marker, rvec, tvec, camera_matrix, camera_dist)
                cam_patch_pts = cam_patch_pts.reshape(-1, 2).astype(np.int32)
                cv2.polylines(display, [cam_patch_pts], True, (255, 255, 0), 2)
            
            tf = tvec.flatten()
            status = f"TRACKING | Z={tf[2]:.0f}mm | ID={mid}"
            status_color = (0, 255, 0)
        else:
            lost_count += 1
            status = f"LOST ({lost_count} frames)"
            status_color = (0, 0, 255)
            # Keep last projection when lost (don't blank out)
            if last_proj_corners is None:
                proj_img = np.zeros((PRJ_H, PRJ_W, 3), dtype=np.uint8)
        
        cv2.imshow("Projector", proj_img)
        
        # Classification
        with torch.no_grad():
            img_tensor = tt(frame).unsqueeze(0).cuda()
            probs = predict_raw(img_tensor)
            pred_class = weights.meta["categories"][probs[0].argmax().item()]
            pred_prob = probs[0].max().item() * 100
            raw_predictions.append(probs)
            pred_prob = probs[0].max().item() * 100
        display = add_text_with_background(display, f'Pred: {pred_class}: {pred_prob:.1f}%',
                                            (10, 10), font_size=24)
        display = add_text_with_background(display, status, (10, 50), font_size=18,
                                            text_color=status_color)
        
        if frame_idx % 30 == 0:
            fps = 30 / ((cv2.getTickCount() - fps_start) / cv2.getTickFrequency())
            fps_start = cv2.getTickCount()
        display = add_text_with_background(display, f'FPS: {fps:.1f} | Frame: {frame_idx}',
                                           (10, 90), font_size=16)
        
        cv2.imshow("Preview", display)
        
        caps.append(frame.copy())
        caps_with_text.append(display)
        results.append(pred_class)
        frame_idx += 1
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('d'):
            show_debug = not show_debug

except KeyboardInterrupt:
    print("\nInterrupted")

cv2.destroyAllWindows()

print(f"\n✓ Captured {len(caps)} frames")

if raw_predictions and 'forbiden_classes' in dir():
    success_count = sum(1 for p in raw_predictions if p[0].argmax().item() not in forbiden_classes)
    success_rate = success_count / len(raw_predictions) * 100
    print(f"  Attack Success Rate: {success_rate:.1f}% ({success_count}/{len(raw_predictions)} frames)")

In [ ]:
frame.shape

## Create GIF

In [ ]:
import imageio

captures = caps
# Create GIF from captures
captures_rgb = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in captures]
output_gif_path = 'tracked_jeep_16x16_1_2025-12-13_16_22_inception_v3.gif'
imageio.mimsave(output_gif_path, captures_rgb, fps=14, loop=0)
print(f"GIF saved to {output_gif_path} with {len(captures)} frames")

In [ ]:
#

## Create MP4

In [ ]:
im_path

In [ ]:
import cv2
import os
import datetime

# Get frame dimensions from first frame
height, width = caps_with_text[0].shape[:2]
fps = 15  # Adjust as needed

# Create output folder with timestamp
cur_time = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
exp_name = os.path.basename(im_path).split(".")[0]
output_folder = os.path.join(r"C:\git\PhysicalAdverserialProj\captured_videos", f"{exp_name}_{cur_time}")
os.makedirs(output_folder, exist_ok=True)

# Create output filename in the new folder
output_video_path = os.path.join(output_folder, f'captured_video.mp4')

# Create VideoWriter with mp4v codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Write all frames
for frame in caps_with_text:
    out.write(frame)

# Release the writer
out.release()

print(f"✓ Video saved to '{output_video_path}'")
print(f"  - Output folder: {output_folder}")
print(f"  - Frames: {len(caps_with_text)}")
print(f"  - Resolution: {width}x{height}")
print(f"  - FPS: {fps}")
print(f"  - Duration: {len(caps_with_text)/fps:.1f} seconds")

## SIFT Tracking with CharuCo Initialization

Alternative tracking approach using SIFT feature matching instead of ArUco marker.
- Initial pose estimation from CharuCo board (placed on projection surface)
- Reference frame captured at initialization
- Movement tracked via SIFT homography relative to reference

In [ ]:
cv2.destroyAllWindows()

## PCA-Based Keypoint Distribution Tracking

**NEW APPROACH**: Instead of matching individual keypoints (which fails due to projection interference):
1. Detect all SIFT keypoints in the YOLO bbox
2. Use PCA to find the principal axes of the keypoint distribution 
3. Track rotation and scale by comparing current axes to reference axes
4. No descriptor matching needed - just compare axis orientation and spread

Benefits:
- Robust to individual keypoint detection noise
- Works even if different keypoints are detected each frame
- Uses aggregate statistics instead of point correspondence

In [ ]:
# ============================================================================
# PCA-BASED KEYPOINT DISTRIBUTION TRACKING
# ============================================================================
# Tracks rotation and scale via keypoint cloud shape, not individual matches
# ============================================================================

import cv2
import numpy as np
import torch
from cv2 import aruco
from scipy.spatial import ConvexHull

# ============================================================================
# CONFIGURATION
# ============================================================================

# CharuCo board configuration - MUST MATCH YOUR PRINTED BOARD
CHARUCO_SQUARES_X = 5
CHARUCO_SQUARES_Y = 4
CHARUCO_SQUARE_SIZE_MM = 45.0
CHARUCO_MARKER_SIZE_MM = 35.0
CHARUCO_DICT_TYPE = aruco.DICT_4X4_100

# Projector settings
PRJ_W, PRJ_H = system.screen_res
PROJECTOR_X_OFFSET = 1920

# SIFT settings
SIFT_CONTRAST_THRESHOLD = 0.03  # Lower = more features
SIFT_EDGE_THRESHOLD = 15.0

# Pose smoothing
POSE_SMOOTH_ALPHA = 0.3

# Get calibration - use already loaded procam_calib from kernel
ref_cam_corners = system.corners_img_proj.reshape(-1, 2).astype(np.float32)
ref_proj_corners = system.orig_proj_corners.reshape(-1, 2).astype(np.float32)

# Stereo calibration - use existing procam_calib dict (not system.procam_calib)
R_stereo = procam_calib['R']  # Rotation from camera to projector
T_stereo = procam_calib['T']  # Translation from camera to projector
# camera_matrix, camera_dist, projector_matrix, projector_dist are already in kernel

# Load patch
patch = cv2.imread(im_path)
if patch is None:
    raise ValueError(f"Could not load patch from {im_path}")
patch_h, patch_w = patch.shape[:2]
src_corners = np.array([[0, 0], [patch_w, 0], [patch_w, patch_h], [0, patch_h]], dtype=np.float32)

# Tensor for classification
tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

print(f"✓ Patch loaded: {im_path}")

# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def create_charuco_board(squares_x, squares_y, square_len, marker_len, aruco_dict):
    if hasattr(aruco, "CharucoBoard"):
        return aruco.CharucoBoard((squares_x, squares_y), square_len, marker_len, aruco_dict)
    return aruco.CharucoBoard_create(squares_x, squares_y, square_len, marker_len, aruco_dict)

def detect_charuco_pose(gray, board, camera_matrix, camera_dist):
    """Detect CharuCo board and estimate pose."""
    if hasattr(aruco, "CharucoDetector"):
        charuco_params = aruco.CharucoParameters()
        detector_params = aruco.DetectorParameters()
        charuco_detector = aruco.CharucoDetector(board, charuco_params, detector_params)
        ch_corners, ch_ids, marker_corners, marker_ids = charuco_detector.detectBoard(gray)
        
        if ch_ids is None or len(ch_ids) < 6:
            return False, None, None, None, None
        
        obj_points, img_points = board.matchImagePoints(ch_corners, ch_ids)
        if len(obj_points) < 6:
            return False, None, None, None, None
        
        success, rvec, tvec = cv2.solvePnP(obj_points, img_points, camera_matrix, camera_dist)
        return success, rvec, tvec, ch_corners, ch_ids
    
    # Older OpenCV fallback
    aruco_dict = board.getDictionary()
    params = aruco.DetectorParameters_create()
    corners, ids, _ = aruco.detectMarkers(gray, aruco_dict, parameters=params)
    
    if ids is None or len(ids) < 4:
        return False, None, None, None, None
    
    _, ch_corners, ch_ids = aruco.interpolateCornersCharuco(corners, ids, gray, board,
                                                             camera_matrix, camera_dist)
    if ch_ids is None or len(ch_ids) < 6:
        return False, None, None, None, None
    
    success, rvec, tvec = cv2.aruco.estimatePoseCharucoBoard(
        ch_corners, ch_ids, board, camera_matrix, camera_dist, None, None
    )
    return success, rvec, tvec, ch_corners, ch_ids

def backproject_to_plane(pixel, camera_matrix, rvec, tvec):
    """Back-project a camera pixel to 3D on the Z=0 plane."""
    R, _ = cv2.Rodrigues(rvec)
    R_inv = R.T
    t_vec = tvec.flatten()
    
    pts_undist = cv2.undistortPoints(
        np.array([[pixel]], dtype=np.float32),
        camera_matrix, None
    )[0, 0]
    
    ray_cam = np.array([pts_undist[0], pts_undist[1], 1.0])
    ray_board = R_inv @ ray_cam
    origin_board = R_inv @ (-t_vec)
    
    if abs(ray_board[2]) < 1e-6:
        return None
    t = -origin_board[2] / ray_board[2]
    pt_3d = origin_board + t * ray_board
    pt_3d[2] = 0
    return pt_3d.astype(np.float32)

def project_to_projector_3d(pts_3d, rvec_cam, tvec_cam):
    """Project 3D points to projector via stereo calibration."""
    R_cam, _ = cv2.Rodrigues(rvec_cam)
    R_proj = R_stereo @ R_cam
    t_proj = (R_stereo @ tvec_cam.flatten() + T_stereo.flatten()).reshape(3, 1)
    rvec_proj, _ = cv2.Rodrigues(R_proj)
    pts, _ = cv2.projectPoints(pts_3d, rvec_proj, t_proj, projector_matrix, projector_dist)
    return pts.reshape(-1, 2).astype(np.float32)

def compute_pca_axes(points):
    """Compute PCA axes from 2D points.
    
    Returns:
        centroid: (x, y) center of mass
        axis1: primary axis direction (unit vector)
        axis2: secondary axis direction (unit vector, perpendicular to axis1)
        spread1: standard deviation along axis1
        spread2: standard deviation along axis2
        angle: rotation angle of axis1 from horizontal (radians)
    """
    if len(points) < 3:
        return None, None, None, 0, 0, 0
    
    points = np.array(points, dtype=np.float64)
    centroid = np.mean(points, axis=0)
    
    # Center the points
    centered = points - centroid
    
    # Compute covariance matrix
    cov = np.cov(centered.T)
    
    # Eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Sort by eigenvalue (largest first)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    axis1 = eigenvectors[:, 0]  # Primary axis
    axis2 = eigenvectors[:, 1]  # Secondary axis
    
    # Ensure consistent axis direction (e.g., axis1 points "right-ish")
    if axis1[0] < 0:
        axis1 = -axis1
    if axis2[1] < 0:
        axis2 = -axis2
    
    spread1 = np.sqrt(eigenvalues[0])  # Standard deviation along axis1
    spread2 = np.sqrt(eigenvalues[1])  # Standard deviation along axis2
    
    angle = np.arctan2(axis1[1], axis1[0])  # Angle in radians
    
    return centroid, axis1, axis2, spread1, spread2, angle

def extract_keypoints_in_bbox(frame_gray, bbox, sift, margin=0.05):
    """Extract SIFT keypoints within a bounding box."""
    x1, y1, x2, y2 = bbox.astype(int)
    w, h = x2 - x1, y2 - y1
    
    # Apply margin
    mx, my = int(w * margin), int(h * margin)
    x1, y1 = x1 + mx, y1 + my
    x2, y2 = x2 - mx, y2 - my
    
    # Detect keypoints
    keypoints, _ = sift.detectAndCompute(frame_gray, None)
    
    # Filter to bbox
    points = []
    for kp in keypoints:
        x, y = kp.pt
        if x1 <= x <= x2 and y1 <= y <= y2:
            points.append([x, y])
    
    return np.array(points) if points else np.array([]).reshape(0, 2)

# ============================================================================
# INITIALIZE
# ============================================================================

# Create SIFT detector
sift = cv2.SIFT_create(
    nfeatures=0,
    contrastThreshold=SIFT_CONTRAST_THRESHOLD,
    edgeThreshold=SIFT_EDGE_THRESHOLD
)

# Create CharuCo board
charuco_dict = aruco.getPredefinedDictionary(CHARUCO_DICT_TYPE)
charuco_board = create_charuco_board(
    CHARUCO_SQUARES_X, CHARUCO_SQUARES_Y,
    CHARUCO_SQUARE_SIZE_MM, CHARUCO_MARKER_SIZE_MM,
    charuco_dict
)

# Window setup
cv2.namedWindow("Projector", cv2.WND_PROP_FULLSCREEN)
cv2.moveWindow("Projector", PROJECTOR_X_OFFSET, 0)
cv2.setWindowProperty("Projector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.namedWindow("Preview", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Preview", 1280, 960)

blank_proj = np.zeros((PRJ_H, PRJ_W, 3), dtype=np.uint8)
cv2.imshow("Projector", blank_proj)

# YOLO for object detection
from ultralytics import YOLO
yolo = YOLO("yolo11n.pt")

print("\n" + "=" * 60)
print("PCA-BASED TRACKING INITIALIZATION")
print("=" * 60)
print("1. Place CharuCo board next to object on same plane")
print("2. Press SPACE when CharuCo & object are visible")
print("3. Select object, then remove CharuCo")
print("=" * 60)

# ============================================================================
# PHASE 1: CharuCo Pose Capture
# ============================================================================

init_rvec = None
init_tvec = None

while True:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    charuco_ok, rvec, tvec, ch_corners, ch_ids = detect_charuco_pose(
        gray, charuco_board, camera_matrix, camera_dist
    )
    
    if charuco_ok:
        cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 50)
        tf = tvec.flatten()
        cv2.putText(display, f"CharuCo OK! Z={tf[2]:.0f}mm", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display, "Press SPACE to capture pose", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    else:
        cv2.putText(display, "Looking for CharuCo board...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    cv2.imshow("Preview", display)
    cv2.imshow("Projector", blank_proj)
    
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        raise KeyboardInterrupt("User quit")
    if key == ord(' ') and charuco_ok:
        init_rvec = rvec.copy()
        init_tvec = tvec.copy()
        print(f"✓ CharuCo pose captured! Distance: {np.linalg.norm(init_tvec):.1f}mm")
        break

# ============================================================================
# PHASE 2: Object Selection
# ============================================================================

print("\n" + "=" * 60)
print("PHASE 2: Select Object")
print("=" * 60)

# Run YOLO on current frame
results = yolo.predict(frame, verbose=False)[0]
if results.boxes is None or len(results.boxes) == 0:
    raise RuntimeError("No objects detected!")

# Draw all detections
boxes_xyxy = results.boxes.xyxy.cpu().numpy()
for i, box in enumerate(boxes_xyxy):
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(display, f"[{i}]", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

cv2.putText(display, "Click on object to track (or press 0-9)", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
cv2.imshow("Preview", display)

# Simple selection - use first detection or wait for key
if len(boxes_xyxy) == 1:
    initial_bbox = boxes_xyxy[0]
    print(f"Auto-selected single detection")
else:
    key = cv2.waitKey(0) & 0xFF
    if ord('0') <= key <= ord('9'):
        idx = key - ord('0')
        if idx < len(boxes_xyxy):
            initial_bbox = boxes_xyxy[idx]
        else:
            initial_bbox = boxes_xyxy[0]
    else:
        initial_bbox = boxes_xyxy[0]

print(f"✓ Selected bbox: {initial_bbox}")

# ============================================================================
# PHASE 3: Reference PCA Capture (with CharuCo still visible or removed)
# ============================================================================

print("\n" + "=" * 60)
print("PHASE 3: Capturing Reference Keypoint Distribution")
print("=" * 60)
print("Remove CharuCo, keep object still, press SPACE")

while True:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Extract keypoints in bbox
    kp_points = extract_keypoints_in_bbox(gray, initial_bbox, sift)
    
    # Draw keypoints
    for pt in kp_points:
        cv2.circle(display, (int(pt[0]), int(pt[1])), 3, (0, 255, 255), -1)
    
    # Draw bbox
    x1, y1, x2, y2 = initial_bbox.astype(int)
    cv2.rectangle(display, (x1, y1), (x2, y2), (255, 0, 0), 2)
    
    # Compute and draw PCA
    if len(kp_points) >= 10:
        centroid, ax1, ax2, sp1, sp2, angle = compute_pca_axes(kp_points)
        
        if centroid is not None:
            cx, cy = int(centroid[0]), int(centroid[1])
            # Draw axes
            end1 = (int(cx + ax1[0]*sp1*2), int(cy + ax1[1]*sp1*2))
            end2 = (int(cx + ax2[0]*sp2*2), int(cy + ax2[1]*sp2*2))
            cv2.arrowedLine(display, (cx, cy), end1, (0, 0, 255), 2, tipLength=0.2)  # Red = axis1
            cv2.arrowedLine(display, (cx, cy), end2, (0, 255, 0), 2, tipLength=0.2)  # Green = axis2
            cv2.circle(display, (cx, cy), 5, (255, 0, 255), -1)  # Magenta centroid
            
            cv2.putText(display, f"KPs: {len(kp_points)} | Angle: {np.degrees(angle):.1f}deg", 
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(display, f"Spread: {sp1:.1f} x {sp2:.1f}", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    else:
        cv2.putText(display, f"Need more keypoints: {len(kp_points)}/10", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    cv2.putText(display, "Press SPACE to capture reference", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    cv2.imshow("Preview", display)
    cv2.imshow("Projector", blank_proj)
    
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        raise KeyboardInterrupt("User quit")
    if key == ord(' ') and len(kp_points) >= 10:
        # Capture reference PCA
        ref_centroid, ref_axis1, ref_axis2, ref_spread1, ref_spread2, ref_angle = compute_pca_axes(kp_points)
        ref_bbox = initial_bbox.copy()
        ref_frame = frame.copy()
        
        # Back-project bbox corners to 3D
        bbox_2d = np.array([
            [initial_bbox[0], initial_bbox[1]],
            [initial_bbox[2], initial_bbox[1]],
            [initial_bbox[2], initial_bbox[3]],
            [initial_bbox[0], initial_bbox[3]]
        ], dtype=np.float32)
        
        object_3d_corners = []
        for pt in bbox_2d:
            pt_3d = backproject_to_plane(pt, camera_matrix, init_rvec, init_tvec)
            if pt_3d is None:
                raise RuntimeError("Back-projection failed")
            object_3d_corners.append(pt_3d)
        object_3d_corners = np.array(object_3d_corners, dtype=np.float32)
        
        obj_width = np.linalg.norm(object_3d_corners[1] - object_3d_corners[0])
        obj_height = np.linalg.norm(object_3d_corners[3] - object_3d_corners[0])
        
        # Compute patch 3D corners (centered in bbox)
        patch_3d_corners = []
        for pt in bbox_2d:
            pt_3d = backproject_to_plane(pt, camera_matrix, init_rvec, init_tvec)
            patch_3d_corners.append(pt_3d)
        patch_3d_corners = np.array(patch_3d_corners, dtype=np.float32)
        
        print(f"✓ Reference captured!")
        print(f"  Centroid: ({ref_centroid[0]:.1f}, {ref_centroid[1]:.1f})")
        print(f"  Angle: {np.degrees(ref_angle):.1f}°")
        print(f"  Spread: {ref_spread1:.1f} x {ref_spread2:.1f}")
        print(f"  Object size: {obj_width:.1f} x {obj_height:.1f} mm")
        break

# Initialize smoothed pose
prev_rvec = init_rvec.copy()
prev_tvec = init_tvec.copy()
smooth_proj_corners = ref_proj_corners.copy()
proj_corners = ref_proj_corners.copy()

# ============================================================================
# PHASE 4: PCA-Based Tracking
# ============================================================================

print("\n" + "=" * 60)
print("PCA TRACKING STARTED")
print("=" * 60)
print("Press 'q' to quit")

frame_idx = 0
fps_start = cv2.getTickCount()
fps = 0
pca_caps = []
pca_results = []

WARMUP_FRAMES = 30

try:
    while True:
        ret, frame = system.cap.read()
        if not ret:
            continue
        
        display = frame.copy()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_idx += 1
        
        # Run YOLO to get current bbox
        results = yolo.predict(frame, verbose=False)[0]
        
        current_bbox = None
        if results.boxes is not None and len(results.boxes) > 0:
            # Use the box closest to previous bbox center
            prev_center = np.array([(ref_bbox[0]+ref_bbox[2])/2, (ref_bbox[1]+ref_bbox[3])/2])
            boxes = results.boxes.xyxy.cpu().numpy()
            best_dist = float('inf')
            for box in boxes:
                center = np.array([(box[0]+box[2])/2, (box[1]+box[3])/2])
                dist = np.linalg.norm(center - prev_center)
                if dist < best_dist:
                    best_dist = dist
                    current_bbox = box
        
        if current_bbox is None:
            current_bbox = ref_bbox  # Fallback
        
        # Extract keypoints
        kp_points = extract_keypoints_in_bbox(gray, current_bbox, sift)
        
        # Draw bbox  
        x1, y1, x2, y2 = current_bbox.astype(int)
        cv2.rectangle(display, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        status = "TRACKING"
        status_color = (0, 255, 0)
        
        if len(kp_points) >= 10:
            # Compute current PCA
            curr_centroid, curr_axis1, curr_axis2, curr_spread1, curr_spread2, curr_angle = compute_pca_axes(kp_points)
            
            if curr_centroid is not None:
                # Draw keypoints and axes
                for pt in kp_points:
                    cv2.circle(display, (int(pt[0]), int(pt[1])), 2, (0, 255, 255), -1)
                
                cx, cy = int(curr_centroid[0]), int(curr_centroid[1])
                end1 = (int(cx + curr_axis1[0]*curr_spread1*2), int(cy + curr_axis1[1]*curr_spread1*2))
                end2 = (int(cx + curr_axis2[0]*curr_spread2*2), int(cy + curr_axis2[1]*curr_spread2*2))
                cv2.arrowedLine(display, (cx, cy), end1, (0, 0, 255), 2, tipLength=0.2)
                cv2.arrowedLine(display, (cx, cy), end2, (0, 255, 0), 2, tipLength=0.2)
                cv2.circle(display, (cx, cy), 5, (255, 0, 255), -1)
                
                # Compute relative rotation and scale
                delta_angle = curr_angle - ref_angle
                scale_x = curr_spread1 / ref_spread1 if ref_spread1 > 0 else 1.0
                scale_y = curr_spread2 / ref_spread2 if ref_spread2 > 0 else 1.0
                scale = (scale_x + scale_y) / 2  # Average scale
                
                # Compute centroid shift in pixels
                centroid_shift = curr_centroid - ref_centroid
                
                # Convert to pose change
                # Rotation around Z-axis (in camera frame) from angle change
                # Scale change indicates distance change (closer = larger)
                
                # Update rvec: add rotation around camera Z
                rvec_update = init_rvec.copy()
                # Add delta rotation (around object's local Z which is roughly camera Z for frontal view)
                R_init, _ = cv2.Rodrigues(init_rvec)
                # Create rotation matrix for delta_angle around Z
                c, s = np.cos(delta_angle), np.sin(delta_angle)
                R_delta = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float64)
                R_new = R_init @ R_delta
                rvec_update, _ = cv2.Rodrigues(R_new)
                
                # Update tvec: scale affects Z (distance)
                tvec_update = init_tvec.copy()
                tvec_update[2] = init_tvec[2] / scale  # Closer = smaller scale factor
                
                # Centroid shift affects X, Y translation
                # Convert pixel shift to mm using depth
                z_mm = tvec_update[2, 0]
                fx, fy = camera_matrix[0, 0], camera_matrix[1, 1]
                shift_x_mm = centroid_shift[0] * z_mm / fx
                shift_y_mm = centroid_shift[1] * z_mm / fy
                tvec_update[0] = init_tvec[0] + shift_x_mm
                tvec_update[1] = init_tvec[1] + shift_y_mm
                
                # Apply smoothing
                smooth_rvec = POSE_SMOOTH_ALPHA * rvec_update + (1 - POSE_SMOOTH_ALPHA) * prev_rvec
                smooth_tvec = POSE_SMOOTH_ALPHA * tvec_update + (1 - POSE_SMOOTH_ALPHA) * prev_tvec
                prev_rvec = smooth_rvec.copy()
                prev_tvec = smooth_tvec.copy()
                
                # Project to projector
                new_proj_corners = project_to_projector_3d(patch_3d_corners, smooth_rvec, smooth_tvec)
                
                # Smooth projection corners
                smooth_proj_corners = POSE_SMOOTH_ALPHA * new_proj_corners + (1 - POSE_SMOOTH_ALPHA) * smooth_proj_corners
                proj_corners = smooth_proj_corners.copy()
                
                # Clamp to screen
                proj_corners[:, 0] = np.clip(proj_corners[:, 0], 0, PRJ_W)
                proj_corners[:, 1] = np.clip(proj_corners[:, 1], 0, PRJ_H)
                
                # Draw axes
                cv2.drawFrameAxes(display, camera_matrix, camera_dist, smooth_rvec, smooth_tvec, 30)
                
                dist = np.linalg.norm(smooth_tvec)
                if frame_idx <= WARMUP_FRAMES:
                    status = f"WARMUP {frame_idx}/{WARMUP_FRAMES}"
                    status_color = (255, 255, 0)
                else:
                    status = f"PCA TRACK | Z:{dist:.0f}mm | Rot:{np.degrees(delta_angle):.1f}° | Scale:{scale:.2f}"
                    status_color = (0, 255, 0)
        else:
            status = f"LOW KPs: {len(kp_points)}"
            status_color = (0, 0, 255)
        
        # Show projection
        if frame_idx <= WARMUP_FRAMES:
            cv2.imshow("Projector", blank_proj)
        else:
            M = cv2.getPerspectiveTransform(src_corners, proj_corners)
            proj_img = cv2.warpPerspective(patch, M, (PRJ_W, PRJ_H))
            cv2.imshow("Projector", proj_img)
        
        # Classification
        with torch.no_grad():
            img_tensor = tt(frame).unsqueeze(0).cuda()
            probs = predict_raw(img_tensor)
            pred_class = weights.meta["categories"][probs[0].argmax().item()]
            pred_prob = probs[0].max().item() * 100
        
        # Overlays
        cv2.putText(display, f"Pred: {pred_class}: {pred_prob:.1f}%", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(display, status, (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        
        if frame_idx % 30 == 0:
            fps = 30 / ((cv2.getTickCount() - fps_start) / cv2.getTickFrequency())
            fps_start = cv2.getTickCount()
            print(f"[Frame {frame_idx}] {status} | FPS: {fps:.1f}")
        
        cv2.putText(display, f"FPS: {fps:.1f} | Frame: {frame_idx}", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow("Preview", display)
        pca_caps.append(frame.copy())
        pca_results.append(pred_class)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

except KeyboardInterrupt:
    print("\nInterrupted")

print(f"\n✓ Captured {len(pca_caps)} frames")
cv2.destroyWindow("Preview")

In [ ]:
# ============================================================================
# HYBRID SIFT TRACKING WITH CHARUCO SCALE + PNP POSE ESTIMATION
# ============================================================================
# Combines:
# - CharuCo for initial scale calibration (get object size in mm)
# - YOLO for object detection
# - SIFT for robust frame-to-frame tracking
# - PnP for 3D pose estimation each frame (resolves homography ambiguity)
# - Stereo projection like ArUco cell 25

import sys
sys.path.insert(0, r"C:\git\Tracker")

import cv2
import cv2.aruco as aruco
import numpy as np
import torch

# Clean up any residual windows from previous runs
cv2.destroyAllWindows()

from tracker import Tracker
from tracker.utils import project_bbox_corners
from classfier_mobilenet import predict_raw, weights

# ============================================================================
# CONFIGURATION
# ============================================================================

# CharuCo board configuration - MUST MATCH YOUR PRINTED BOARD
CHARUCO_SQUARES_X = 5
CHARUCO_SQUARES_Y = 4
CHARUCO_SQUARE_SIZE_MM = 45.0
CHARUCO_MARKER_SIZE_MM = 35.0
CHARUCO_DICT_TYPE = aruco.DICT_4X4_100

# Tracker configuration
YOLO_MODEL = "yolo11n.pt"
MIN_INLIERS = 15  # Lowered for more robust tracking
SIFT_RATIO = 0.9  # Very lenient matching (default 0.75)
KEYFRAME_INTERVAL = 15  # Create keyframe every 15 frames for faster adaptation
BBOX_MARGIN = 0.05  # Shrink bbox by 5% to exclude background keypoints
CENTER_MARGIN = 0.2  # Exclude center 60% where projection lands

# Pose smoothing (EMA filter)
POSE_SMOOTH_ALPHA = 0.3  # Lower = smoother but more lag, Higher = faster but more jitter (0.0-1.0)

# Get calibration corners from system
ref_cam_corners = system.corners_img_proj.reshape(-1, 2).astype(np.float32)
ref_proj_corners = system.orig_proj_corners.reshape(-1, 2).astype(np.float32)

# Load adversarial patch
patch = cv2.imread(im_path)
if patch is None:
    raise ValueError(f"Could not load patch from {im_path}")
patch_h, patch_w = patch.shape[:2]
src_corners = np.array([[0, 0], [patch_w, 0], [patch_w, patch_h], [0, patch_h]], dtype=np.float32)

# Tensor conversion
tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

print(f"✓ Patch loaded: {im_path} ({patch_w}x{patch_h})")

# ============================================================================
# CHARUCO FUNCTIONS
# ============================================================================

def create_charuco_board(squares_x, squares_y, square_len, marker_len, aruco_dict):
    if hasattr(aruco, "CharucoBoard"):
        return aruco.CharucoBoard((squares_x, squares_y), square_len, marker_len, aruco_dict)
    return aruco.CharucoBoard_create(squares_x, squares_y, square_len, marker_len, aruco_dict)

def detect_charuco_pose(gray, board, camera_matrix, camera_dist):
    """Detect CharuCo board and estimate pose."""
    if hasattr(aruco, "CharucoDetector"):
        charuco_params = aruco.CharucoParameters()
        detector_params = aruco.DetectorParameters()
        charuco_detector = aruco.CharucoDetector(board, charuco_params, detector_params)
        ch_corners, ch_ids, marker_corners, marker_ids = charuco_detector.detectBoard(gray)
        
        if ch_ids is None or len(ch_ids) < 6:
            return False, None, None, None, None
        
        obj_points, img_points = board.matchImagePoints(ch_corners, ch_ids)
        if len(obj_points) < 6:
            return False, None, None, None, None
        
        success, rvec, tvec = cv2.solvePnP(obj_points, img_points, camera_matrix, camera_dist)
        return success, rvec, tvec, ch_corners, ch_ids
    
    # Older OpenCV fallback
    aruco_dict = board.getDictionary()
    params = aruco.DetectorParameters_create()
    corners, ids, _ = aruco.detectMarkers(gray, aruco_dict, parameters=params)
    
    if ids is None or len(ids) < 4:
        return False, None, None, None, None
    
    _, ch_corners, ch_ids = aruco.interpolateCornersCharuco(corners, ids, gray, board,
                                                             camera_matrix, camera_dist)
    if ch_ids is None or len(ch_ids) < 6:
        return False, None, None, None, None
    
    success, rvec, tvec = cv2.aruco.estimatePoseCharucoBoard(
        ch_corners, ch_ids, board, camera_matrix, camera_dist, None, None
    )
    return success, rvec, tvec, ch_corners, ch_ids

def backproject_to_plane(pixel, camera_matrix, rvec, tvec):
    """Back-project a camera pixel to 3D on the Z=0 plane (CharuCo board surface)."""
    R, _ = cv2.Rodrigues(rvec)
    R_inv = R.T
    t_vec = tvec.flatten()
    
    # Undistort point
    pts_undist = cv2.undistortPoints(
        np.array([[pixel]], dtype=np.float32),
        camera_matrix, None
    )[0, 0]
    
    # Ray in camera frame
    ray_cam = np.array([pts_undist[0], pts_undist[1], 1.0])
    
    # Transform to board frame
    ray_board = R_inv @ ray_cam
    origin_board = R_inv @ (-t_vec)
    
    # Intersect with Z=0 plane
    if abs(ray_board[2]) < 1e-6:
        return None
    t = -origin_board[2] / ray_board[2]
    pt_3d = origin_board + t * ray_board
    pt_3d[2] = 0
    return pt_3d.astype(np.float32)

def project_to_projector_3d(pts_3d, rvec_cam, tvec_cam):
    """Project 3D points to projector via stereo calibration."""
    R_cam, _ = cv2.Rodrigues(rvec_cam)
    R_proj = R_stereo @ R_cam
    t_proj = (R_stereo @ tvec_cam.flatten() + T_stereo.flatten()).reshape(3, 1)
    rvec_proj, _ = cv2.Rodrigues(R_proj)
    pts, _ = cv2.projectPoints(pts_3d, rvec_proj, t_proj, projector_matrix, projector_dist)
    return pts.reshape(-1, 2).astype(np.float32)

# ============================================================================
# INITIALIZATION
# ============================================================================

# Create CharuCo board
charuco_dict = aruco.getPredefinedDictionary(CHARUCO_DICT_TYPE)
charuco_board = create_charuco_board(
    CHARUCO_SQUARES_X, CHARUCO_SQUARES_Y,
    CHARUCO_SQUARE_SIZE_MM, CHARUCO_MARKER_SIZE_MM,
    charuco_dict
)

import importlib
import tracker
import tracker.tracker as tracker_module
import tracker.utils as tracker_utils
importlib.reload(tracker_utils)  # Reload utils module first
importlib.reload(tracker_module)  # Reload the actual module with Tracker class
importlib.reload(tracker)  # Reload parent to pick up changes
from tracker import Tracker

tracker = Tracker(
    yolo_model_path=YOLO_MODEL,
    sift_nfeatures=5000,
    sift_ratio_threshold=SIFT_RATIO,
    min_inliers=MIN_INLIERS,
    use_affine=False,
    homography_ransac_threshold=5.0,
    bbox_margin=BBOX_MARGIN,  # Shrink bbox to exclude background
    center_margin=CENTER_MARGIN  # Exclude center where projection lands
)

# Window setup
cv2.namedWindow("Projector", cv2.WND_PROP_FULLSCREEN)
cv2.moveWindow("Projector", PROJECTOR_X_OFFSET, 0)
cv2.setWindowProperty("Projector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
cv2.namedWindow("Preview", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Preview", 1280, 960)

# Show blank during initialization
blank_proj = np.zeros((PRJ_H, PRJ_W, 3), dtype=np.uint8)
cv2.imshow("Projector", blank_proj)

print("\n" + "=" * 60)
print("HYBRID SIFT+CHARUCO INITIALIZATION")
print("=" * 60)
print("1. Place CharuCo board next to object on same plane")
print("2. Press SPACE when both CharuCo & object are visible")
print("3. Select object with YOLO detection")
print("Press 'q' to quit")
print("=" * 60)

# ============================================================================
# PHASE 1: CharuCo + YOLO Initialization (Two-step)
# ============================================================================

init_rvec = None
init_tvec = None
object_3d_corners = None  # 4x3 array in mm
initial_bbox = None

# Step 1: Capture CharuCo pose
print("\n" + "=" * 60)
print("STEP 1: CharuCo Pose Capture")
print("=" * 60)
print("Place CharuCo board next to object on same plane")
print("Press SPACE when CharuCo is detected")
print("=" * 60)

while True:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect CharuCo
    charuco_ok, rvec, tvec, ch_corners, ch_ids = detect_charuco_pose(
        gray, charuco_board, camera_matrix, camera_dist
    )
    
    if charuco_ok:
        cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 50)
        tf = tvec.flatten()
        cv2.putText(display, f"CharuCo OK! Z={tf[2]:.0f}mm", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display, "Press SPACE to capture pose", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    else:
        cv2.putText(display, "Looking for CharuCo board...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    cv2.imshow("Preview", display)
    cv2.imshow("Projector", blank_proj)
    
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        cv2.destroyWindow("Preview")
        raise KeyboardInterrupt("User quit during initialization")
    
    if key == ord(' ') and charuco_ok:
        init_rvec = rvec.copy()
        init_tvec = tvec.copy()
        print(f"✓ CharuCo pose captured! Distance: {np.linalg.norm(init_tvec):.1f}mm")
        break

# Step 2: Wait for clear scene (CharuCo removed)
print("\n" + "=" * 60)
print("STEP 2: Clear Scene for SIFT Initialization")
print("=" * 60)
print("Remove CharuCo board from scene")
print("Keep object visible")
print("Press SPACE when coast is clear")
print("=" * 60)

while True:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    
    # Check if CharuCo is still visible (should not be)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    charuco_ok, _, _, _, _ = detect_charuco_pose(
        gray, charuco_board, camera_matrix, camera_dist
    )
    
    if charuco_ok:
        cv2.putText(display, "CharuCo still visible - remove it!", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    else:
        cv2.putText(display, "CharuCo removed - press SPACE when ready", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.putText(display, "Object should be visible without annotations", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    cv2.imshow("Preview", display)
    cv2.imshow("Projector", blank_proj)
    
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        cv2.destroyWindow("Preview")
        raise KeyboardInterrupt("User quit during initialization")
    
    if key == ord(' '):
        # Ensure projector is showing black
        cv2.imshow("Projector", blank_proj)
        cv2.waitKey(100)  # Wait for projector to update
        
        # Capture multiple frames for robust keypoint aggregation
        INIT_FRAMES = 100  # Number of frames to capture for initialization
        print(f"\n⏳ Capturing {INIT_FRAMES} frames for robust keypoint collection...")
        init_frames = []
        
        for i in range(INIT_FRAMES):
            ret, f = system.cap.read()
            if ret:
                init_frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            cv2.waitKey(10)  # ~100fps capture
            
            # Show progress
            if (i + 1) % 20 == 0:
                display = f.copy() if ret else display
                cv2.putText(display, f"Capturing: {i+1}/{INIT_FRAMES}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                cv2.imshow("Preview", display)
        
        clean_frame = cv2.cvtColor(init_frames[-1], cv2.COLOR_RGB2BGR)  # Keep last frame for display
        print(f"✓ Captured {len(init_frames)} frames for multi-frame initialization")
        break

# Step 3: Run YOLO detection and multi-frame SIFT initialization
print("\nRunning YOLO detection...")
clean_frame_rgb = init_frames[-1]  # Use last captured frame
detection_output = r"C:\git\PhysicalAdverserialProj\yolo_detections.jpg"

# First get the bbox using YOLO
result = tracker.yolo_tracker.model.predict(clean_frame_rgb, verbose=False)[0]
if result.boxes is None or len(result.boxes) == 0:
    cv2.destroyWindow("Preview")
    raise RuntimeError("No objects detected")

# Select object
from tracker.yolo_tracker import xywh_to_xyxy
boxes_xywh = result.boxes.xywh.cpu().numpy()
if len(boxes_xywh) == 1:
    initial_bbox = xywh_to_xyxy(boxes_xywh[0])
else:
    selection = tracker.yolo_tracker.select_object_interactive(clean_frame_rgb, detection_output)
    if selection is None:
        cv2.destroyWindow("Preview")
        raise RuntimeError("No object selected")
    _, initial_bbox = selection

tracker.current_bbox = initial_bbox.copy()
tracker.bboxes.append(initial_bbox.copy())

print(f"✓ Selected bbox: {initial_bbox}")

# Use multi-frame initialization for robust keypoints
print("\n⏳ Aggregating keypoints from multiple frames...")
num_agg_keypoints = tracker.initialize_reference_multi_frame(
    init_frames, 
    initial_bbox,
    spatial_threshold=3.0  # Cluster keypoints within 3 pixels
)
print(f"✓ Multi-frame initialization complete: {num_agg_keypoints} aggregated keypoints")

# Back-project bbox corners to 3D on CharuCo plane
x1, y1, x2, y2 = initial_bbox
bbox_2d = np.array([
    [x1, y1],  # top-left
    [x2, y1],  # top-right
    [x2, y2],  # bottom-right
    [x1, y2]   # bottom-left
], dtype=np.float32)

object_3d_corners = []
for pt in bbox_2d:
    pt_3d = backproject_to_plane(pt, camera_matrix, init_rvec, init_tvec)
    if pt_3d is None:
        cv2.destroyWindow("Preview")
        raise RuntimeError("Back-projection failed")
    object_3d_corners.append(pt_3d)
obj_height = np.linalg.norm(object_3d_corners[3] - object_3d_corners[0])
object_3d_corners = np.array(object_3d_corners, dtype=np.float32)

# Compute object size
obj_width = np.linalg.norm(object_3d_corners[1] - object_3d_corners[0])
obj_height = np.linalg.norm(object_3d_corners[3] - object_3d_corners[0])

print(f"✓ Object 3D corners computed:")
print(f"  Size: {obj_width:.1f} x {obj_height:.1f} mm")
print(f"  Distance: {np.linalg.norm(init_tvec):.1f} mm")

# DEBUG: Show and save reference keypoints
ref_keypoints = tracker.keyframes[0].keypoints
print(f"✓ Reference keypoints: {len(ref_keypoints)}")

# Draw reference keypoints on clean frame
debug_frame = clean_frame.copy()
for kp in ref_keypoints:
    x, y = int(kp.pt[0]), int(kp.pt[1])
    cv2.circle(debug_frame, (x, y), 4, (0, 255, 0), -1)

# Draw bbox
cv2.rectangle(debug_frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
cv2.putText(debug_frame, f"{len(ref_keypoints)} keypoints", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

# Save debug image
debug_path = r"C:\git\PhysicalAdverserialProj\debug_reference_keypoints.jpg"
cv2.imwrite(debug_path, debug_frame)
print(f"✓ Saved reference keypoints to: {debug_path}")

# Show for a moment
cv2.imshow("Preview", debug_frame)
cv2.waitKey(1000)

if object_3d_corners is None:
    cv2.destroyWindow("Preview")
    raise RuntimeError("Initialization failed")

# Use object_3d_corners for patch projection (patch covers the object!)
# patch_3d_corners should be the SAME as object_3d_corners since we want 
# to project the patch onto the object, not onto calibration area
patch_3d_corners = object_3d_corners.copy()
print(f"✓ Patch 3D corners (from object bbox): {patch_3d_corners.shape}")
print(f"  Size: {np.linalg.norm(patch_3d_corners[1] - patch_3d_corners[0]):.1f} x {np.linalg.norm(patch_3d_corners[3] - patch_3d_corners[0]):.1f} mm")

# Store previous pose for temporal consistency
prev_rvec = init_rvec.copy()
prev_tvec = init_tvec.copy()

# ============================================================================
# PHASE 2: SIFT Tracking with PnP
# ============================================================================

print("\n" + "=" * 60)
print("TRACKING STARTED")
print("=" * 60)
print("Press 'q' to quit, 'r' to re-initialize")

sift_caps = []
sift_caps_with_text = []
sift_results = []
frame_idx = 1
fps_start = cv2.getTickCount()
fps = 0
proj_corners = ref_proj_corners.copy()
smooth_proj_corners = ref_proj_corners.copy()  # For EMA smoothed projection

# Warm-up period: track with black projection first to stabilize
WARMUP_FRAMES = 30  # Track with black projection for first 30 frames
print(f"⏳ Warm-up: Tracking {WARMUP_FRAMES} frames with BLACK projection...")

try:
    while True:
        ret, frame = system.cap.read()
        if not ret:
            continue
        display = frame.copy()
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Track frame using SIFT
        H = tracker.track_frame(frame_rgb, frame_idx)
        
        # =================================================================
        # INLIER DISAPPEARANCE DIAGNOSTIC
        # =================================================================
        if len(tracker.frame_stats) >= 2:
            curr_stats = tracker.frame_stats[-1]
            prev_stats = tracker.frame_stats[-2]
            
            curr_inliers = curr_stats.get('num_inliers', 0)
            prev_inliers = prev_stats.get('num_inliers', 0)
            inlier_drop = prev_inliers - curr_inliers
            
            # Detect sudden drop (more than 50% or drop to near-zero)
            if prev_inliers > 10 and (inlier_drop > prev_inliers * 0.5 or curr_inliers < 5):
                print(f"\n{'='*60}")
                print(f"⚠️ SUDDEN INLIER DROP DETECTED at frame {frame_idx}")
                print(f"{'='*60}")
                print(f"  Inliers: {prev_inliers} → {curr_inliers} (dropped {inlier_drop})")
                print(f"  Status: {curr_stats.get('status', '?')} | Reason: {curr_stats.get('failure_reason', 'N/A')}")
                print(f"  Matches: {curr_stats.get('num_matches', 0)}")
                print(f"  Current keypoints: {len(curr_stats.get('current_keypoints', []))}")
                print(f"  Reference keypoints: {curr_stats.get('num_ref_keypoints', '?')}")
                
                # Match quality info
                avg_dist = curr_stats.get('avg_match_distance', 0)
                max_dist = curr_stats.get('max_match_distance', 0)
                prev_avg_dist = prev_stats.get('avg_match_distance', 0)
                print(f"  Match distance: avg={avg_dist:.1f} (was {prev_avg_dist:.1f}), max={max_dist:.1f}")
                
                # Check bbox movement
                if len(tracker.bboxes) >= 2:
                    curr_bbox = tracker.bboxes[-1]
                    prev_bbox = tracker.bboxes[-2]
                    bbox_shift = np.linalg.norm(curr_bbox - prev_bbox)
                    bbox_size_change = abs((curr_bbox[2]-curr_bbox[0])*(curr_bbox[3]-curr_bbox[1]) - 
                                          (prev_bbox[2]-prev_bbox[0])*(prev_bbox[3]-prev_bbox[1]))
                    print(f"  Bbox shift: {bbox_shift:.1f}px, size change: {bbox_size_change:.0f}px²")
                    if bbox_shift > 50:
                        print(f"  → YOLO bbox JUMPED significantly!")
                
                # Check keyframe being used
                print(f"  Keyframe used: {curr_stats.get('keyframe_idx', '?')} (of {len(tracker.keyframes)} keyframes)")
                
                # Analyze spatial distribution of keypoints
                if 'current_keypoints' in curr_stats and 'keyframe_keypoints' in curr_stats:
                    curr_kps = curr_stats['current_keypoints']
                    ref_kps = curr_stats.get('keyframe_keypoints', [])
                    if len(curr_kps) > 0:
                        curr_pts = np.array([kp.pt for kp in curr_kps])
                        curr_centroid = np.mean(curr_pts, axis=0)
                        curr_std = np.std(curr_pts, axis=0)
                        print(f"  Current kps: centroid=({curr_centroid[0]:.0f},{curr_centroid[1]:.0f}), spread=({curr_std[0]:.0f},{curr_std[1]:.0f})")
                    if len(ref_kps) > 0:
                        ref_pts = np.array([kp.pt for kp in ref_kps])
                        ref_centroid = np.mean(ref_pts, axis=0)
                        ref_std = np.std(ref_pts, axis=0)
                        print(f"  Reference kps: centroid=({ref_centroid[0]:.0f},{ref_centroid[1]:.0f}), spread=({ref_std[0]:.0f},{ref_std[1]:.0f})")
                        
                # Inlier spatial analysis
                if curr_stats.get('inlier_mask') is not None and curr_stats.get('matches') is not None:
                    inlier_mask = curr_stats['inlier_mask']
                    matches = curr_stats['matches']
                    num_outliers = np.sum(~inlier_mask) if hasattr(inlier_mask, '__len__') else 0
                    print(f"  Outliers rejected by RANSAC: {num_outliers}")
                    
                # Show recent history (last 5 frames before failure)
                print("  Recent 5-frame history:")
                history_start = max(0, len(tracker.frame_stats) - 6)
                for i, hist_stats in enumerate(tracker.frame_stats[history_start:-1]):
                    h_inliers = hist_stats.get('num_inliers', 0)
                    h_matches = hist_stats.get('num_matches', 0)
                    h_status = 'OK' if hist_stats.get('status') == 'success' else 'FAIL'
                    h_dist = hist_stats.get('avg_match_distance', 0)
                    print(f"    [-{5-i}] inliers={h_inliers}/{h_matches} dist={h_dist:.0f} {h_status}")
                    
                print(f"{'='*60}\n")
        
        # DEBUG: Draw current keypoints from tracking stats
        if tracker.frame_stats:
            stats = tracker.frame_stats[-1]
            if 'current_keypoints' in stats and stats['current_keypoints'] is not None:
                curr_kps = stats['current_keypoints']
                for kp in curr_kps:
                    x, y = int(kp.pt[0]), int(kp.pt[1])
                    cv2.circle(display, (x, y), 3, (0, 255, 255), -1)  # Yellow for current
            
            # Show inlier/outlier info
            num_inliers = stats.get('num_inliers', 0)
            num_matches = stats.get('num_matches', 0)
            status_str = stats.get('status', 'unknown')
            cv2.putText(display, f"Inliers: {num_inliers}/{num_matches} | Status: {status_str}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        
        # DEBUG: Draw YOLO bbox (blue) - this shows where SIFT keypoints are filtered
        if tracker.current_bbox is not None:
            yolo_bbox = tracker.current_bbox.astype(np.int32)
            cv2.rectangle(display, (yolo_bbox[0], yolo_bbox[1]), (yolo_bbox[2], yolo_bbox[3]), 
                         (255, 0, 0), 2)  # Blue for YOLO bbox
            cv2.putText(display, "YOLO", (yolo_bbox[0], yolo_bbox[1]-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            
            # Draw effective bbox with margins (cyan = outer shrink, magenta = center exclusion)
            x1, y1, x2, y2 = yolo_bbox
            width = x2 - x1
            height = y2 - y1
            outer_margin_x = int(width * BBOX_MARGIN)
            outer_margin_y = int(height * BBOX_MARGIN)
            center_margin_x = int(width * CENTER_MARGIN)
            center_margin_y = int(height * CENTER_MARGIN)
            # Outer boundary (keypoints must be inside this)
            cv2.rectangle(display, (x1 + outer_margin_x, y1 + outer_margin_y), 
                         (x2 - outer_margin_x, y2 - outer_margin_y),
                         (255, 255, 0), 1)  # Cyan for outer boundary
            # Center exclusion zone (keypoints must be outside this)
            cv2.rectangle(display, (x1 + center_margin_x, y1 + center_margin_y),
                         (x2 - center_margin_x, y2 - center_margin_y),
                         (255, 0, 255), 1)  # Magenta for center exclusion
        
        # Show number of keyframes
        cv2.putText(display, f"Keyframes: {len(tracker.keyframes)}", (10, 150),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        
        pose_ok = False
        if H is not None:
            # Project bbox corners to current frame using homography
            curr_bbox_corners = project_bbox_corners(initial_bbox, H)
            
            # Draw tracked bbox
            bbox_pts = curr_bbox_corners.astype(np.int32)
            cv2.polylines(display, [bbox_pts], True, (0, 255, 0), 3)
            
            # Solve PnP with known 3D object size
            success, rvec, tvec = cv2.solvePnP(
                object_3d_corners,
                curr_bbox_corners.astype(np.float32),
                camera_matrix,
                camera_dist,
                rvec=prev_rvec.copy(),
                tvec=prev_tvec.copy(),
                useExtrinsicGuess=True,
                flags=cv2.SOLVEPNP_ITERATIVE
            )
            
            if success:
                pose_ok = True
                
                # --- JUMP REJECTION: Reject sudden large changes in pose ---
                MAX_TVEC_JUMP = 100  # mm - max allowed translation jump between frames
                MAX_RVEC_JUMP = 0.5  # radians - max allowed rotation jump
                
                tvec_jump = np.linalg.norm(tvec - prev_tvec)
                rvec_jump = np.linalg.norm(rvec - prev_rvec)
                
                if tvec_jump > MAX_TVEC_JUMP or rvec_jump > MAX_RVEC_JUMP:
                    print(f"[Frame {frame_idx}] ⚠️ JUMP REJECTED: tvec_jump={tvec_jump:.1f}mm, rvec_jump={rvec_jump:.3f}rad")
                    # Use previous pose instead
                    smooth_rvec = prev_rvec.copy()
                    smooth_tvec = prev_tvec.copy()
                else:
                    # Apply EMA smoothing to reduce jitter
                    # smoothed = alpha * new + (1 - alpha) * old
                    smooth_rvec = POSE_SMOOTH_ALPHA * rvec + (1 - POSE_SMOOTH_ALPHA) * prev_rvec
                    smooth_tvec = POSE_SMOOTH_ALPHA * tvec + (1 - POSE_SMOOTH_ALPHA) * prev_tvec
                    
                    # Update for next frame
                    prev_rvec = smooth_rvec.copy()
                    prev_tvec = smooth_tvec.copy()
                
                # Project patch corners to projector using smoothed pose
                new_proj_corners = project_to_projector_3d(
                    patch_3d_corners, smooth_rvec, smooth_tvec
                )
                
                # Check for NaN or extreme values
                if np.any(np.isnan(new_proj_corners)) or np.any(np.abs(new_proj_corners) > 10000):
                    print(f"[Frame {frame_idx}] ⚠️ Invalid projection corners detected!")
                    print(f"  tvec: {smooth_tvec.flatten()}")
                    print(f"  rvec: {smooth_rvec.flatten()}")
                    # Skip this frame's projection update
                else:
                    # Also smooth the projection corners for stable output
                    smooth_proj_corners = POSE_SMOOTH_ALPHA * new_proj_corners + (1 - POSE_SMOOTH_ALPHA) * smooth_proj_corners
                    
                    # --- BOUNDS CLAMPING: Keep projection within projector screen ---
                    MARGIN = 50  # Allow slight overshoot
                    clamped = smooth_proj_corners.copy()
                    clamped[:, 0] = np.clip(clamped[:, 0], -MARGIN, PRJ_W + MARGIN)
                    clamped[:, 1] = np.clip(clamped[:, 1], -MARGIN, PRJ_H + MARGIN)
                    
                    proj_corners = clamped.copy()
                
                # Draw coordinate axes with smoothed pose
                cv2.drawFrameAxes(display, camera_matrix, camera_dist, smooth_rvec, smooth_tvec, 30)
                
                dist = np.linalg.norm(smooth_tvec)
                if frame_idx <= WARMUP_FRAMES:
                    status = f"WARMUP {frame_idx}/{WARMUP_FRAMES} | Dist: {dist:.0f}mm"
                    status_color = (255, 255, 0)  # Yellow during warmup
                else:
                    status = f"TRACKING | Dist: {dist:.0f}mm"
                    status_color = (0, 255, 0)
            else:
                status = "PNP FAILED"
                status_color = (0, 0, 255)
        else:
            # SIFT tracking failed
            status = "SIFT LOST"
            status_color = (0, 0, 255)
        
        if not pose_ok:
            stats = tracker.frame_stats[-1] if tracker.frame_stats else {}
            inliers = stats.get('num_inliers', 0)
        else:
            stats = tracker.frame_stats[-1] if tracker.frame_stats else {}
            inliers = stats.get('num_inliers', 0)
        
        # Create and show projection
        # During warmup: show black. After warmup: show patch
        if frame_idx <= WARMUP_FRAMES:
            # Warmup phase - black projection
            cv2.imshow("Projector", blank_proj)
            if frame_idx == WARMUP_FRAMES:
                print(f"\n✓ Warmup complete! Starting adversarial projection...")
        else:
            # Active phase - show patch
            # Debug: Check if projection corners are valid
            proj_min = proj_corners.min(axis=0)
            proj_max = proj_corners.max(axis=0)
            proj_in_bounds = (proj_min[0] >= 0 and proj_min[1] >= 0 and 
                              proj_max[0] <= PRJ_W and proj_max[1] <= PRJ_H)
            
            if frame_idx % 30 == 0 or not proj_in_bounds:
                print(f"[Frame {frame_idx}] Projection corners:")
                print(f"  TL: ({proj_corners[0][0]:.0f}, {proj_corners[0][1]:.0f})")
                print(f"  TR: ({proj_corners[1][0]:.0f}, {proj_corners[1][1]:.0f})")
                print(f"  BR: ({proj_corners[2][0]:.0f}, {proj_corners[2][1]:.0f})")
                print(f"  BL: ({proj_corners[3][0]:.0f}, {proj_corners[3][1]:.0f})")
                print(f"  In bounds: {proj_in_bounds} (PRJ: {PRJ_W}x{PRJ_H})")
                if not proj_in_bounds:
                    print(f"  ⚠️ PROJECTION OFF SCREEN!")
            
            M = cv2.getPerspectiveTransform(src_corners, proj_corners)
            proj_img = cv2.warpPerspective(patch, M, (PRJ_W, PRJ_H))
            cv2.imshow("Projector", proj_img)
        
        # Classification
        with torch.no_grad():
            img_tensor = tt(frame).unsqueeze(0).cuda()
            probs = predict_raw(img_tensor)
            pred_class = weights.meta["categories"][probs[0].argmax().item()]
            pred_prob = probs[0].max().item() * 100
        
        # Text overlays
        cv2.putText(display, f"Pred: {pred_class}: {pred_prob:.1f}%", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(display, status, (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        
        if frame_idx % 30 == 0:
            fps = 30 / ((cv2.getTickCount() - fps_start) / cv2.getTickFrequency())
            fps_start = cv2.getTickCount()
            # Enhanced debug output
            stats = tracker.frame_stats[-1] if tracker.frame_stats else {}
            status_str = stats.get('status', 'unknown')
            yolo_ok = "YES" if tracker.current_bbox is not None else "NO"
            failure_reason = stats.get('failure_reason', '')
            num_kps = len(stats.get('current_keypoints', [])) if stats.get('current_keypoints') is not None else 0
            num_matches = stats.get('num_matches', 0)
            num_ref = stats.get('num_ref_keypoints', 0)
            avg_dist = stats.get('avg_match_distance', 0)
            
            debug_msg = f"[Frame {frame_idx}] Inliers: {inliers}/{num_matches}, KPs: {num_kps}/{num_ref}, Dist: {avg_dist:.0f}, {status_str}"
            if failure_reason:
                debug_msg += f", FAIL: {failure_reason}"
            print(debug_msg)
        cv2.putText(display, f"FPS: {fps:.1f} | Frame: {frame_idx}", (10, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow("Preview", display)
        
        sift_caps.append(frame.copy())
        sift_caps_with_text.append(display)
        sift_results.append(pred_class)
        frame_idx += 1
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            print("\nRe-initialization not yet implemented for hybrid mode")


except KeyboardInterrupt:
    print("\nInterrupted")

print(f"\n✓ Captured {len(sift_caps)} frames")

cv2.destroyWindow("Preview")

## Post capture analysis

In [ ]:
# Setup for analysis
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2
import pickle
import importlib

# Import orig_clases - vehicle classes we want to avoid
import consts
importlib.reload(consts)
orig_clases = consts.orig_clases

from aruco_pose import get_camera_angles_from_frame

# Load camera calibration for ArUco detection
cameraMatrix, dist_coeffs = pickle.load(open(r"C:\git\wall_alignment\calibration.pkl", "rb"))

tt = lambda x: torch.tensor(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)/255.).permute(2,0,1).float()

print(f"Original vehicle classes to avoid: {orig_clases.tolist()}")
print(f"Number of captured frames: {len(caps)}")

In [ ]:
# Load all classifiers for evaluation
print("Loading classifiers...")

# Ensemble v1 classifier (Inception, ResNet, VGG, ViT, DINOv2)
import classfier_ensemble as ensemble_v1
ensemble_v1_weights = ensemble_v1.weights
print("✓ Ensemble v1 loaded (Inception, ResNet, VGG, ViT, DINOv2)")

# Ensemble v2 classifier (ConvNeXt, EfficientNet, MobileNet, Swin)
import classfier_ensemble_v2 as ensemble_v2
ensemble_v2_weights = ensemble_v2.weights
print("✓ Ensemble v2 loaded (ConvNeXt, EfficientNet, MobileNet, Swin)")

# DINOv2 standalone classifier
import classfier_dino as dino
dino_weights = dino.weights
print("✓ DINOv2 standalone classifier loaded")

categories = ensemble_v1_weights.meta["categories"]
print(f"\n✓ All classifiers ready for evaluation")

In [ ]:
# Analyze all captured frames - extract pose info and classify with all models
results_list = []

for frame_idx, frame in enumerate(tqdm(caps, desc="Analyzing frames")):
    # Get ArUco pose (angle and distance)
    pose_result = get_camera_angles_from_frame(frame)
    
    if not pose_result['found']:
        continue  # Skip frames without ArUco marker
    
    angle = pose_result['angle']
    distance = pose_result['distance_m']
    
    # Prepare image tensor
    img_tensor = tt(frame).unsqueeze(0).cuda()
    
    with torch.no_grad():
        # Get predictions from each model in ensemble v1
        ensemble_v1_per_model = ensemble_v1.predict_raw_per_model(img_tensor)
        
        # Get predictions from each model in ensemble v2
        ensemble_v2_per_model = ensemble_v2.predict_raw_per_model(img_tensor)
        
        # Get DINOv2 standalone prediction
        dino_probs = dino.predict_raw(img_tensor)
    
    # Build result row
    result_row = {
        'frame_idx': frame_idx,
        'angle_deg': angle,
        'distance_m': distance,
    }
    
    # Process ensemble v1 models (individual) - Inception, ResNet, VGG, ViT, DINOv2
    for model_name in ['inception', 'resnet', 'vgg', 'vit', 'dino']:
        if model_name in ensemble_v1_per_model:
            probs = ensemble_v1_per_model[model_name]
            pred_idx = probs[0].argmax().item()
            pred_conf = probs[0][pred_idx].item()
            pred_class = categories[pred_idx]
            
            # Check if attack is successful (prediction NOT in orig_clases)
            is_success = pred_idx not in orig_clases.tolist()
            
            result_row[f'v1_{model_name}_class'] = pred_class
            result_row[f'v1_{model_name}_idx'] = pred_idx
            result_row[f'v1_{model_name}_conf'] = pred_conf
            result_row[f'v1_{model_name}_success'] = is_success
    
    # Ensemble v1 combined prediction
    if 'ensemble' in ensemble_v1_per_model:
        ens_probs = ensemble_v1_per_model['ensemble']
        ens_pred_idx = ens_probs[0].argmax().item()
        ens_pred_conf = ens_probs[0][ens_pred_idx].item()
        ens_pred_class = categories[ens_pred_idx]
        ens_is_success = ens_pred_idx not in orig_clases.tolist()
        
        result_row['v1_combined_class'] = ens_pred_class
        result_row['v1_combined_idx'] = ens_pred_idx
        result_row['v1_combined_conf'] = ens_pred_conf
        result_row['v1_combined_success'] = ens_is_success
    
    # Process ensemble v2 models (individual) - ConvNeXt, EfficientNet, MobileNet, Swin
    for model_name in ['convnext', 'efficientnet', 'mobilenet', 'swin']:
        if model_name in ensemble_v2_per_model:
            probs = ensemble_v2_per_model[model_name]
            pred_idx = probs[0].argmax().item()
            pred_conf = probs[0][pred_idx].item()
            pred_class = categories[pred_idx]
            
            # Check if attack is successful (prediction NOT in orig_clases)
            is_success = pred_idx not in orig_clases.tolist()
            
            result_row[f'v2_{model_name}_class'] = pred_class
            result_row[f'v2_{model_name}_idx'] = pred_idx
            result_row[f'v2_{model_name}_conf'] = pred_conf
            result_row[f'v2_{model_name}_success'] = is_success
    
    # Ensemble v2 combined prediction
    if 'ensemble' in ensemble_v2_per_model:
        ens_probs = ensemble_v2_per_model['ensemble']
        ens_pred_idx = ens_probs[0].argmax().item()
        ens_pred_conf = ens_probs[0][ens_pred_idx].item()
        ens_pred_class = categories[ens_pred_idx]
        ens_is_success = ens_pred_idx not in orig_clases.tolist()
        
        result_row['v2_combined_class'] = ens_pred_class
        result_row['v2_combined_idx'] = ens_pred_idx
        result_row['v2_combined_conf'] = ens_pred_conf
        result_row['v2_combined_success'] = ens_is_success
    
    # DINOv2 standalone
    dino_pred_idx = dino_probs[0].argmax().item()
    dino_pred_conf = dino_probs[0][dino_pred_idx].item()
    dino_pred_class = categories[dino_pred_idx]
    dino_is_success = dino_pred_idx not in orig_clases.tolist()
    
    result_row['dino_standalone_class'] = dino_pred_class
    result_row['dino_standalone_idx'] = dino_pred_idx
    result_row['dino_standalone_conf'] = dino_pred_conf
    result_row['dino_standalone_success'] = dino_is_success
    
    results_list.append(result_row)

print(f"\n✓ Analyzed {len(results_list)} frames with ArUco marker detected")
df = pd.DataFrame(results_list)

In [ ]:
%matplotlib inline

In [ ]:
# Calculate and display success rates per model with clear v1/v2/dino distinction
print("="*80)
print("ATTACK SUCCESS RATE ANALYSIS")
print("="*80)
print(f"\nSuccess = classifier output NOT in orig_clases (vehicle classes)")
print(f"Total frames analyzed: {len(df)}")
print(f"Angle range: {df['angle_deg'].min():.1f}° to {df['angle_deg'].max():.1f}°")
print(f"Distance range: {df['distance_m'].min():.2f}m to {df['distance_m'].max():.2f}m")

# Get all success columns
success_cols = [c for c in df.columns if c.endswith('_success')]

# Define model groups for clear display - DINOv2 standalone only
v1_models = ['v1_inception', 'v1_resnet', 'v1_vgg', 'v1_vit', 'v1_combined']
dino_models = ['dino_standalone']  # Only standalone DINOv2
v2_models = ['v2_convnext', 'v2_efficientnet', 'v2_mobilenet', 'v2_swin', 'v2_combined']

# Display names mapping
display_names = {
    'v1_inception': 'Inception V3',
    'v1_resnet': 'ResNet18',
    'v1_vgg': 'VGG16',
    'v1_vit': 'ViT-B/16',
    'v1_combined': '*** COMBINED ***',
    'v2_convnext': 'ConvNeXt Base',
    'v2_efficientnet': 'EfficientNet B0',
    'v2_mobilenet': 'MobileNetV3 Large',
    'v2_swin': 'Swin Transformer',
    'v2_combined': '*** COMBINED ***',
    'dino_standalone': 'DINOv2',
}

success_summary = []

def print_model_results(model_list, df, success_summary, group_name):
    for model_name in model_list:
        success_col = f'{model_name}_success'
        if success_col in df.columns:
            success_count = df[success_col].sum()
            total_count = len(df)
            success_rate = success_count / total_count * 100
            
            display = display_names.get(model_name, model_name)
            print(f"  {display:<25} {success_rate:>10.1f}% {success_count:>10} / {total_count:<8}")
            
            success_summary.append({
                'Model': display,
                'Group': group_name,
                'Success Rate (%)': success_rate,
                'Successful Attacks': int(success_count),
                'Total Frames': total_count
            })

# Print Ensemble v1 results
print(f"\n{'─'*80}")
print("ENSEMBLE V1 (Inception V3, ResNet18, VGG16, ViT-B/16)")
print(f"{'─'*80}")
print(f"  {'Model':<25} {'Success Rate':>10} {'Successful':>10} {'Total':>8}")
print_model_results(v1_models, df, success_summary, 'V1')

# Print DINOv2 results (standalone only)
print(f"\n{'─'*80}")
print("DINOv2 CLASSIFIER")
print(f"{'─'*80}")
print(f"  {'Model':<25} {'Success Rate':>10} {'Successful':>10} {'Total':>8}")
print_model_results(dino_models, df, success_summary, 'DINOv2')

# Print Ensemble v2 results
print(f"\n{'─'*80}")
print("ENSEMBLE V2 (ConvNeXt, EfficientNet, MobileNet, Swin Transformer)")
print(f"{'─'*80}")
print(f"  {'Model':<25} {'Success Rate':>10} {'Successful':>10} {'Total':>8}")
print_model_results(v2_models, df, success_summary, 'V2')

success_df = pd.DataFrame(success_summary)
print("\n" + "="*80)

In [ ]:
# Visualize success rate vs angle and distance
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Create angle bins for analysis
angle_bins = np.linspace(df['angle_deg'].min(), df['angle_deg'].max(), 11)
df['angle_bin'] = pd.cut(df['angle_deg'], bins=angle_bins)

# Create distance bins for analysis
distance_bins = np.linspace(df['distance_m'].min(), df['distance_m'].max(), 6)
df['distance_bin'] = pd.cut(df['distance_m'], bins=distance_bins)

# Compute angle_mids once for reuse
angle_success_v1 = df.groupby('angle_bin', observed=False)['v1_combined_success'].agg(['mean', 'count'])
angle_mids = [(b.left + b.right)/2 for b in angle_success_v1.index]

# 1. Overall success rate bar chart
ax = axes[0, 0]
colors = ['green' if r > 50 else 'orange' if r > 25 else 'red' for r in success_df['Success Rate (%)']]
bars = ax.barh(success_df['Model'], success_df['Success Rate (%)'], color=colors)
ax.set_xlabel('Success Rate (%)')
ax.set_title('Attack Success Rate by Model')
ax.set_xlim(0, 100)
ax.axvline(x=50, color='black', linestyle='--', alpha=0.5, label='50% threshold')
for i, (bar, rate) in enumerate(zip(bars, success_df['Success Rate (%)'])):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{rate:.1f}%', va='center')

# 2. Success rate vs Angle (V1 ensemble combined)
ax = axes[0, 1]
angle_success_v1['mean'] = angle_success_v1['mean'] * 100
ax.bar(range(len(angle_mids)), angle_success_v1['mean'], color='steelblue', alpha=0.7)
ax.set_xticks(range(len(angle_mids)))
ax.set_xticklabels([f'{a:.0f}°' for a in angle_mids], rotation=45)
ax.set_xlabel('Viewing Angle (degrees)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('V1 Combined: Success Rate vs Angle')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5)

# 3. Success rate vs Distance (V1 ensemble combined)
ax = axes[0, 2]
dist_success = df.groupby('distance_bin', observed=False)['v1_combined_success'].agg(['mean', 'count'])
dist_success['mean'] = dist_success['mean'] * 100
dist_mids = [(b.left + b.right)/2 for b in dist_success.index]
ax.bar(range(len(dist_mids)), dist_success['mean'], color='darkorange', alpha=0.7)
ax.set_xticks(range(len(dist_mids)))
ax.set_xticklabels([f'{d:.2f}m' for d in dist_mids], rotation=45)
ax.set_xlabel('Distance (meters)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('V1 Combined: Success Rate vs Distance')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5)

# 4. Scatter plot: Angle vs Distance colored by success (V1 ensemble)
ax = axes[1, 0]
colors_scatter = ['green' if s else 'red' for s in df['v1_combined_success']]
ax.scatter(df['angle_deg'], df['distance_m'], c=colors_scatter, alpha=0.5, s=20)
ax.set_xlabel('Viewing Angle (degrees)')
ax.set_ylabel('Distance (meters)')
ax.set_title('V1 Combined: Success Distribution\n(Green=Success, Red=Fail)')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)

# 5. Success rate vs Angle for DINOv2 standalone
ax = axes[1, 1]
angle_success_dino = df.groupby('angle_bin', observed=False)['dino_standalone_success'].agg(['mean', 'count'])
angle_success_dino['mean'] = angle_success_dino['mean'] * 100
ax.bar(range(len(angle_mids)), angle_success_dino['mean'], color='purple', alpha=0.7)
ax.set_xticks(range(len(angle_mids)))
ax.set_xticklabels([f'{a:.0f}°' for a in angle_mids], rotation=45)
ax.set_xlabel('Viewing Angle (degrees)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('DINOv2 Standalone: Success Rate vs Angle')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5)

# 6. Model comparison across angle bins (V1 + V2 + DINOv2)
ax = axes[1, 2]
model_keys = ['v1_inception', 'v1_resnet', 'v1_vgg', 'v1_vit', 'v1_combined', 'v2_combined', 'dino_standalone']
model_display = ['Inception', 'ResNet', 'VGG', 'ViT', 'V1 Comb', 'V2 Comb', 'DINOv2']
x = np.arange(len(angle_mids))
width = 0.11
for i, (key, display) in enumerate(zip(model_keys, model_display)):
    if f'{key}_success' in df.columns:
        model_angle_success = df.groupby('angle_bin', observed=False)[f'{key}_success'].mean() * 100
        ax.bar(x + i*width - width*3, model_angle_success.values, width, label=display, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'{a:.0f}°' for a in angle_mids], rotation=45)
ax.set_xlabel('Viewing Angle (degrees)')
ax.set_ylabel('Success Rate (%)')
ax.set_title('All Models: Success Rate vs Angle')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('attack_success_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Plot saved to 'attack_success_analysis.png'")

In [ ]:
# Detailed breakdown: What classes do models predict when attack succeeds?
print("="*80)
print("ATTACK SUCCESS BREAKDOWN - Top Misclassifications")
print("="*80)

# Define model names based on success columns
success_cols = [c for c in df.columns if c.endswith('_success')]
model_names = [c.replace('_success', '') for c in success_cols]

for model_name, success_col in zip(model_names, success_cols):
    class_col = model_name + '_class'
    if class_col in df.columns:
        # Filter successful attacks only
        successful_df = df[df[success_col] == True]
        
        if len(successful_df) > 0:
            top_misclassifications = successful_df[class_col].value_counts().head(5)
            display_name = model_name.replace('v1_', 'V1: ').replace('v2_', 'V2: ').replace('_', ' ').title()
            
            print(f"\n{display_name} - Successful attacks ({len(successful_df)} frames):")
            print(f"  Top predicted classes (non-vehicle):")
            for cls, count in top_misclassifications.items():
                pct = count / len(successful_df) * 100
                print(f"    - {cls}: {count} ({pct:.1f}%)")

print("\n" + "="*80)

In [ ]:
# Save analysis results to CSV
output_csv = 'attack_success_analysis_results.csv'
df.to_csv(output_csv, index=False)

# Save summary to CSV
summary_csv = 'attack_success_summary.csv'
success_df.to_csv(summary_csv, index=False)

print(f"✓ Detailed results saved to '{output_csv}'")
print(f"✓ Summary saved to '{summary_csv}'")

# Display summary table
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(success_df.to_string(index=False))

In [ ]:
import pandas as pd
results_top = pd.DataFrame(results,columns=['class']).groupby('class').size().sort_values(ascending=False).head(5)

In [ ]:
list(results_top.index)

In [ ]:
categories.index('backpack')

In [ ]:
# forbiden_classes = [636,748,414]
forbiden_classes = [817, 705, 609, 586, 436, 627, 468, 621, 803, 407, 408, 751, 717,866, 661, 864]

In [ ]:
# Create video with fixed class probability histogram overlay (smoothed)
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Patch
import os
import datetime
from collections import deque

# Get categories from weights
categories = weights.meta["categories"]

fixed_class_names = ['jeep',  'half track' ,'digital clock', 'golfcart']
# Fixed classes to show (in order): mailbag, soap dispenser, iron, washbasin, bathtub
# fixed_class_names = ['purse','mailbag','backpack' ,'soap dispenser', 'washbasin', 'bathtub']
# fixed_class_names = ['banana'] + ['bathtub',  'soap dispenser', 'washbasin', 'toilet seat']

fixed_indices = []
for name in fixed_class_names:
    try:
        idx = categories.index(name)
        fixed_indices.append(idx)
        print(f"Found '{name}' at index {idx}")
    except ValueError:
        print(f"Warning: '{name}' not found in categories")
        fixed_indices.append(None)

# Video settings
height, width = caps[0].shape[:2]
fps = 15
hist_width = 200  # Width of histogram overlay (halved from 400)
hist_height = 100  # Height of histogram overlay (halved from 200)

# Output path
cur_time = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
exp_name = 'jeep_tracked'#os.path.basename(im_path).split(".")[0]
output_folder = os.path.join(r"C:\git\PhysicalAdverserialProj\captured_videos", f"{exp_name}_histogram_{cur_time}")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(output_folder, 'captured_video_with_histogram.mp4')

# Create VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Smoothing: exponential moving average
SMOOTHING_ALPHA = 0.3  # Lower = smoother, higher = more responsive
smoothed_probs = None

def create_histogram_overlay(probs_np, class_names, forbidden_classes, fixed_indices, size=(200, 100)):
    """Create a histogram image showing fixed classes"""
    # Get probabilities for fixed classes
    display_probs = [probs_np[idx] if idx is not None else 0.0 for idx in fixed_indices]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(size[0]/100, size[1]/100), dpi=100)
    fig.patch.set_facecolor('black')
    fig.patch.set_alpha(0.7)
    ax.set_facecolor('black')
    
    # Color bars: red for forbidden (vehicle) classes, green for successful attack
    colors = ['#ff4444' if (idx is not None and idx in forbidden_classes) else '#44ff44' for idx in fixed_indices]
    
    # Create horizontal bar chart
    y_pos = np.arange(len(class_names))
    bars = ax.barh(y_pos, [p * 100 for p in display_probs], color=colors, edgecolor='white', linewidth=0.5)
    
    # Add probability text on bars
    for i, (bar, prob) in enumerate(zip(bars, display_probs)):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                f'{prob*100:.1f}%', va='center', ha='left', color='white', fontsize=6, fontweight='bold')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(class_names, color='white', fontsize=7)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Probability (%)', color='white', fontsize=6)
    ax.tick_params(axis='x', colors='white', labelsize=5)
    ax.set_title('Class Probabilities', color='white', fontsize=8, fontweight='bold')
    
    # Add legend for True class/False class
    legend_elements = [Patch(facecolor='#44ff44', edgecolor='white', label='False class'),
                       Patch(facecolor='#ff4444', edgecolor='white', label='True class')]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=5, 
              facecolor='black', edgecolor='white', labelcolor='white')
    
    plt.tight_layout(pad=0.3)
    
    # Convert to image
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    hist_img = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)
    hist_img = hist_img.reshape(canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    return cv2.cvtColor(hist_img, cv2.COLOR_RGB2BGR)

# Process each frame
print(f"Creating video with histogram overlay...")
print(f"Total frames: {len(caps)}")
print(f"Smoothing alpha: {SMOOTHING_ALPHA} (lower = smoother)")

for frame_idx, (frame, probs) in enumerate(zip(caps, raw_predictions_mobilenet)):
    probs_np = probs[0].cpu().numpy()
    
    # Apply exponential moving average smoothing
    if smoothed_probs is None:
        smoothed_probs = probs_np.copy()
    else:
        smoothed_probs = SMOOTHING_ALPHA * probs_np + (1 - SMOOTHING_ALPHA) * smoothed_probs
    
    # Create histogram overlay with smoothed probabilities
    hist_img = create_histogram_overlay(smoothed_probs, fixed_class_names, forbiden_classes, fixed_indices, size=(hist_width, hist_height))
    
    # Resize histogram to fit
    hist_img = cv2.resize(hist_img, (hist_width, hist_height))
    
    # Overlay histogram on frame (top-left corner)
    frame_with_hist = frame.copy()
    x_offset = 10
    y_offset = 10
    
    # Blend histogram with frame (less opaque: 50% frame + 50% histogram)
    roi = frame_with_hist[y_offset:y_offset+hist_height, x_offset:x_offset+hist_width]
    blended = cv2.addWeighted(roi, 0.5, hist_img, 0.5, 0)
    frame_with_hist[y_offset:y_offset+hist_height, x_offset:x_offset+hist_width] = blended
    
    out.write(frame_with_hist)
    
    if frame_idx % 100 == 0:
        print(f"  Processed {frame_idx}/{len(caps)} frames...")

out.release()

# Calculate overall success rate
success_count = sum(1 for p in raw_predictions if p[0].argmax().item() not in forbiden_classes)
success_rate = success_count / len(raw_predictions) * 100

print(f"\n✓ Video saved to '{output_video_path}'")
print(f"  - Frames: {len(caps)}")
print(f"  - Resolution: {width}x{height}")
print(f"  - FPS: {fps}")
print(f"  - Duration: {len(caps)/fps:.1f} seconds")
print(f"\n  Attack Success Rate: {success_rate:.1f}% ({success_count}/{len(raw_predictions)} frames)")

In [ ]:
len(caps)

In [ ]:
len(raw_predictions)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(caps[0])

In [ ]:
# =============================================================================
# Load saved captures from pkl files
# =============================================================================
import pickle

load_path = r"C:\git\PhysicalAdverserialProj\infer_caps_dir\5_1\2026-01-19_14_40"

with open(f"{load_path}/caps.pkl", "rb") as f:
    caps = pickle.load(f)
with open(f"{load_path}/caps_with_text.pkl", "rb") as f:
    caps_with_text = pickle.load(f)
with open(f"{load_path}/results.pkl", "rb") as f:
    results = pickle.load(f)
with open(f"{load_path}/raw_predictions.pkl", "rb") as f:
    raw_predictions = pickle.load(f)

print(f"✓ Loaded data from: {load_path}")
print(f"  - caps: {len(caps)} frames")
print(f"  - caps_with_text: {len(caps_with_text)} frames")
print(f"  - results: {len(results)} predictions")
print(f"  - raw_predictions: {len(raw_predictions)} tensors")